In [1]:
# %pip install torch

In [2]:
# %pip install pandas numpy  

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GraphSAGE, global_mean_pool

In [4]:
THREAT_PATH = r"E:\insider detection system threat\insider_4_2_final.csv"
# THREAT_PATH ="E:\insider detection system threat\insider_4_2_final.csv"
threats = pd.read_csv(THREAT_PATH)
threats["user"]  = threats["user"].str.strip()
threats["start"] = pd.to_datetime(threats["start"])
threats["end"]   = pd.to_datetime(threats["end"])

# Normalise type label: 1=Exfil, 2=Sabotage
threats["threat_type"] = threats["what type of threat"].str.strip().str.lower()
threats["type_id"] = threats["threat_type"].map(
    lambda x: 1 if "exfil" in x else 2
)
print(threats[["user","start","end","type_id"]].head())
print("end")

      user               start                 end  type_id
0  AAM0658 2010-10-23 01:34:19 2010-10-29 05:23:28        1
1  AJR0932 2010-09-10 19:12:01 2010-09-18 02:02:51        1
2  BDV0168 2010-07-30 19:56:44 2010-08-10 05:16:41        1
3  BIH0745 2010-07-13 20:15:23 2010-07-13 21:20:44        1
4  BLS0678 2010-09-21 01:16:22 2010-09-30 04:48:19        1
end


In [5]:
def load_csv(name):
    df = pd.read_csv(f"{BASE}/{name}.csv")
    df.columns = df.columns.str.lower()
    df["user"] = df["user"].str.strip()
    df["date"] = pd.to_datetime(df["date"])
    return df


# BASE = "E:\insider detection system threat\r4.2"
BASE = r"E:\insider detection system threat\r4.2"

logon  = load_csv("logon")
print("over ......")
file   = load_csv("file")
print("over ......")
email  = load_csv("email")
print("over ......")


over ......
over ......
over ......


In [6]:
def basic_analysis(df, name):
    print(f"\n{name.upper()} DATASET")
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nMissing values:")
    print(df.isnull().sum())

In [7]:
devices = load_csv("device")

In [8]:
http   = load_csv("http")
print("over ......")

over ......


In [9]:
threats.info()

<class 'pandas.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   details              70 non-null     str           
 1   user                 70 non-null     str           
 2   start                70 non-null     datetime64[us]
 3   end                  70 non-null     datetime64[us]
 4   label                70 non-null     int64         
 5   what type of threat  70 non-null     str           
 6   threat_type          70 non-null     str           
 7   type_id              70 non-null     int64         
dtypes: datetime64[us](2), int64(2), str(4)
memory usage: 4.5 KB


In [10]:
# %pip install glob

In [11]:
import glob
def load_ldap(path):
    df = pd.read_csv(path)

    # standardize column names
    df.columns = df.columns.str.lower()

    # clean user column
    df["user_id"] = df["user_id"].str.strip()

    # convert date column if exists
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])

    return df


# ── Load all LDAP monthly files ────────────────────────────────────
ldap_files = sorted(glob.glob(f"{BASE}/LDAP/*.csv"))

ldap_all = pd.concat(
    [load_ldap(f) for f in ldap_files],
    ignore_index=True
)

# keep latest record for each user (captures role/department updates)
ldap_user = ldap_all.groupby("user_id").last().reset_index()

print("LDAP shape:", ldap_user.shape)
print("LDAP columns:", ldap_user.columns.tolist())


LDAP shape: (1000, 9)
LDAP columns: ['user_id', 'employee_name', 'email', 'role', 'business_unit', 'functional_unit', 'department', 'team', 'supervisor']


In [12]:
ldap_user.head()

,user_id,employee_name,email,role,business_unit,functional_unit,department,team,supervisor
0,AAE0190,August Armando Evans,August.Armando.Evans@dtaa.com,Manager,1,6 - PurchasingAndContracts,1 - Purchasing,NaN,Kirby Bo Pollard
1,AAF0535,Athena Amelia Foreman,Athena.Amelia.Foreman@dtaa.com,Technician,1,5 - SalesAndMarketing,3 - FieldService,2 - RegionalFieldService,Burke Randall Burnett
2,AAF0791,Aladdin Abraham Foley,Aladdin.Abraham.Foley@dtaa.com,Physicist,1,2 - ResearchAndEngineering,1 - Research,5 - Lab,Sasha Britanney Quinn
3,AAL0706,April Alika Levy,April.Alika.Levy@dtaa.com,ProductionLineWorker,1,3 - Manufacturing,3 - Assembly,4 - AssemblyDept,Alan Benjamin Holder
4,AAM0658,Abel Adam Morton,Abel.Adam.Morton@dtaa.com,Technician,1,5 - SalesAndMarketing,3 - FieldService,3 - RegionalFieldService,Hedwig Regina Livingston


In [13]:
def basic_analysis(df, name):
    print(f"\n{'='*40}")
    print(f"{name.upper()} DATASET")
    print(f"{'='*40}")
    print("Shape:", df.shape)
    print("\nColumns:")
    for col in df.columns:
        print(f" - {col} ({df[col].dtype})")
    print("\nMissing values:")
    display(df.isnull().sum().to_frame("missing_count"))
for d, n in zip([logon, file, email, http, devices],
        ["logon", "file", "email", "http", "device"]):
    basic_analysis(d, n)


LOGON DATASET
Shape: (854859, 5)

Columns:
 - id (str)
 - date (datetime64[us])
 - user (str)
 - pc (str)
 - activity (str)

Missing values:


,missing_count
id,0
date,0
user,0
pc,0
activity,0



FILE DATASET
Shape: (445581, 6)

Columns:
 - id (str)
 - date (datetime64[us])
 - user (str)
 - pc (str)
 - filename (str)
 - content (str)

Missing values:


,missing_count
id,0
date,0
user,0
pc,0
filename,0
content,0



EMAIL DATASET
Shape: (2629979, 11)

Columns:
 - id (str)
 - date (datetime64[us])
 - user (str)
 - pc (str)
 - to (str)
 - cc (str)
 - bcc (str)
 - from (str)
 - size (int64)
 - attachments (int64)
 - content (str)

Missing values:


,missing_count
id,0
date,0
user,0
pc,0
to,0
cc,1617054
bcc,2212977
from,0
size,0
attachments,0



HTTP DATASET
Shape: (28434423, 6)

Columns:
 - id (str)
 - date (datetime64[us])
 - user (str)
 - pc (str)
 - url (str)
 - content (str)

Missing values:


,missing_count
id,0
date,0
user,0
pc,0
url,0
content,0



DEVICE DATASET
Shape: (405380, 5)

Columns:
 - id (str)
 - date (datetime64[us])
 - user (str)
 - pc (str)
 - activity (str)

Missing values:


,missing_count
id,0
date,0
user,0
pc,0
activity,0


In [14]:
email["cc"]  = email["cc"].fillna("")
email["bcc"] = email["bcc"].fillna("")


In [15]:
basic_analysis(email,"email")



EMAIL DATASET
Shape: (2629979, 11)

Columns:
 - id (str)
 - date (datetime64[us])
 - user (str)
 - pc (str)
 - to (str)
 - cc (str)
 - bcc (str)
 - from (str)
 - size (int64)
 - attachments (int64)
 - content (str)

Missing values:


,missing_count
id,0
date,0
user,0
pc,0
to,0
cc,0
bcc,0
from,0
size,0
attachments,0


In [16]:
from sklearn.preprocessing import LabelEncoder
print("LDAP Feature Engineering")
ldap_enc = ldap_user[['user_id']].copy()

categorical_cols = ['role','dept','team','ITAdmin']

for col in categorical_cols:
    if col in ldap_user.columns:
        le = LabelEncoder()
        ldap_enc[f'ldap_{col}'] = le.fit_transform(
            ldap_user[col].fillna('Unknown')
        )
    else:
        ldap_enc[f'ldap_{col}'] = 0


# ITAdmin flag
if 'role' in ldap_user.columns:
    ldap_enc['is_ITAdmin'] = ldap_user['role'].str.contains(
        'itadmin', case=False, na=False
    ).astype(np.int8)
else:
    ldap_enc['is_ITAdmin'] = 0


# supervisor feature
if 'supervisor' in ldap_user.columns:

    sup_counts = (
        ldap_user.groupby('supervisor')
        .size()
        .rename('n_reports')
        .reset_index()
    )

    sup_counts.columns = ['user_id','n_reports']

    ldap_enc = ldap_enc.merge(
        sup_counts,
        on='user_id',
        how='left'
    )

    ldap_enc['n_reports'] = ldap_enc['n_reports'].fillna(0)

else:
    ldap_enc['n_reports'] = 0


LDAP_COLS = [c for c in ldap_enc.columns if c!='user_id']

print("LDAP columns:", LDAP_COLS)

print(ldap_enc.head())
LDAP_COLS = [c for c in ldap_enc.columns if c != 'user_id']
print('LDAP feature cols:', LDAP_COLS)


LDAP Feature Engineering
LDAP columns: ['ldap_role', 'ldap_dept', 'ldap_team', 'ldap_ITAdmin', 'is_ITAdmin', 'n_reports']
   user_id  ldap_role  ldap_dept  ldap_team  ldap_ITAdmin  is_ITAdmin  \
0  AAE0190         22          0         38             0           0   
1  AAF0535         39          0         11             0           0   
2  AAF0791         28          0         28             0           0   
3  AAL0706         30          0         21             0           0   
4  AAM0658         39          0         18             0           0   

   n_reports  
0        0.0  
1        0.0  
2        0.0  
3        0.0  
4        0.0  
LDAP feature cols: ['ldap_role', 'ldap_dept', 'ldap_team', 'ldap_ITAdmin', 'is_ITAdmin', 'n_reports']


In [17]:
WINDOW_HOURS = 12

def create_windows(df):
    df = df.copy()
    df["window"] = df["date"].dt.floor(f"{WINDOW_HOURS}h")
    return df

logon  = create_windows(logon)
devices = create_windows(devices)
http   = create_windows(http)
email  = create_windows(email)
file   = create_windows(file)

print("Log groups ready")

Log groups ready


In [18]:
logon.head()


,id,date,user,pc,activity,window
0,{X1D9-S0ES98JV-5357PWMI},2010-01-02 06:49:00,NGF0157,PC-6056,Logon,2010-01-02
1,{G2B3-L6EJ61GT-2222RKSO},2010-01-02 06:50:00,LRR0148,PC-4275,Logon,2010-01-02
2,{U6Q3-U0WE70UA-3770UREL},2010-01-02 06:53:04,LRR0148,PC-4124,Logon,2010-01-02
3,{I0N5-R7NA26TG-6263KNGM},2010-01-02 07:00:00,IRM0931,PC-7188,Logon,2010-01-02
4,{D1S0-N6FH62BT-5398KANK},2010-01-02 07:00:00,MOH0273,PC-6699,Logon,2010-01-02


In [19]:
file.head()


,id,date,user,pc,filename,content,window
0,{L9G8-J9QE34VM-2834VDPB},2010-01-02 07:23:14,MOH0273,PC-6699,EYPC9Y08.doc,D0-CF-11-E0-A1-B1-1A-E1 during difficulty over...,2010-01-02
1,{H0W6-L4FG38XG-9897XTEN},2010-01-02 07:26:19,MOH0273,PC-6699,N3LTSU3O.pdf,25-50-44-46-2D carpenters 25 landed strait dis...,2010-01-02
2,{M3Z0-O2KK89OX-5716MBIM},2010-01-02 08:12:03,HPH0075,PC-2417,D3D3WC9W.doc,D0-CF-11-E0-A1-B1-1A-E1 union 24 declined impo...,2010-01-02
3,{E1I4-S4QS61TG-3652YHKR},2010-01-02 08:17:00,HPH0075,PC-2417,QCSW62YS.doc,D0-CF-11-E0-A1-B1-1A-E1 becoming period begin ...,2010-01-02
4,{D4R7-E7JL45UX-0067XALT},2010-01-02 08:24:57,HSB0196,PC-8001,AU75JV6U.jpg,FF-D8,2010-01-02


In [20]:
base = pd.concat([
    logon[['user','window']],
    devices[['user','window']],
    http[['user','window']],
    email[['user','window']],
    file[['user','window']]
], ignore_index=True).drop_duplicates()

base.head()

,user,window
0,NGF0157,2010-01-02
1,LRR0148,2010-01-02
3,IRM0931,2010-01-02
4,MOH0273,2010-01-02
5,LAP0338,2010-01-02


In [21]:
# 1. Pre-calculate boolean flags (Faster than doing it inside the groupby)
devices["is_connect"] = devices["activity"].eq("connect")
devices["is_disconnect"] = devices["activity"].eq("disconnect")
devices["is_after_hours"] = ((devices["date"].dt.hour < 9) | (devices["date"].dt.hour >= 17)).astype(int)

# 2. Single Groupby Aggregation
devices_features = devices.groupby(["user", "window"]).agg(
    devices_count=('activity', 'size'),
    devices_connect=('is_connect', 'sum'),
    devices_disconnect=('is_disconnect', 'sum'),
    devices_ah=('is_after_hours', 'sum')
)

# 3. Calculate missing disconnects via vectorized subtraction on the resulting columns
devices_features["devices_miss_dc"] = (
    devices_features["devices_connect"] - devices_features["devices_disconnect"]).clip(lower=0)

In [22]:
http_features = http.copy()

# create small flags first
http_features["is_ah"] = (
    (http_features["date"].dt.hour < 9) |
    (http_features["date"].dt.hour >= 17)
).astype("uint8")

http_features["is_we"] = (
    http_features["date"].dt.dayofweek >= 5
).astype("uint8")

http_features["is_we_ah"] = (
    http_features["is_ah"] & http_features["is_we"]
).astype("uint8")

# extract domain without exploding memory
http_features["domain"] = http_features["url"].str.extract(
    r"https?://([^/]+)"
)


In [23]:
SUSP_KEYWORDS = {
    'leak', 'wikileaks', 'whistleblower', 'anonymous', 'darkweb', 'tor',
    'upload', 'dropbox', 'pastebin', 'mega', 'job', 'linkedin', 'resume',
    'career', 'glassdoor', 'competitor','ambitionbox','indeed','naukari','ats'
}


In [24]:
import numpy as np
import pandas as pd

# 1. Process Keywords (Your O(1) logic is great)
keywords = set(k.lower() for k in SUSP_KEYWORDS)

def count_keywords(text):
    if not isinstance(text, str):
        return 0
    words = set(text.lower().split())
    return 1 if not words.isdisjoint(keywords) else 0

# 2. Attach the flag directly to http_features
# Using list comprehension for speed as you suggested
http_features["is_susp"] = np.array(
    [count_keywords(x) for x in http["content"]], 
    dtype='uint8'
)

# 3. Perform a SINGLE aggregation
# This is much more memory efficient than grouping twice
http_features_agg = http_features.groupby(["user", "window"]).agg(
    http_count=("url", "size"),
    http_ah=("is_ah", "sum"),
    http_we=("is_we", "sum"),
    http_we_ah=("is_we_ah", "sum"),
    http_susp_count=("is_susp", "sum"),      # Now this works!
    http_distinct_domains=("domain", "nunique") 
).reset_index()

print(http_features_agg.head())

      user              window  http_count  http_ah  http_we  http_we_ah  \
0  AAE0190 2010-01-04 00:00:00          62        9        0           0   
1  AAE0190 2010-01-04 12:00:00          81       22        0           0   
2  AAE0190 2010-01-05 00:00:00          57       17        0           0   
3  AAE0190 2010-01-05 12:00:00          86       21        0           0   
4  AAE0190 2010-01-06 00:00:00          69       25        0           0   

   http_susp_count  http_distinct_domains  
0                2                     34  
1                3                     40  
2                1                     33  
3                1                     37  
4                1                     27  


In [25]:
# http_susp_agg

In [26]:
# http_features_agg = http_features.groupby(["user", "window"]).agg(
#     http_count=("url", "size"),
#     http_ah=("is_ah", "sum"),
#     http_we=("is_we", "sum"),
#     http_we_ah=("is_we_ah", "sum"),
#     http_susp_count=("is_susp", "sum"),      # Now this works!
#     http_distinct_domains=("domain", "nunique") 
# ).reset_index()

# print(http_features_agg.head())

In [27]:
logon["is_after_hours"] = (
    (logon["date"].dt.hour < 9) | (logon["date"].dt.hour >= 17)
).astype("uint8")  # ✅ uint8 saves memory vs int64

logon["is_weekend"] = (logon["date"].dt.dayofweek >= 5).astype("uint8")
logon["hour"] = logon["date"].dt.hour.astype("uint8")

# Shared PC flag
pc_users = logon.groupby("pc")["user"].nunique()
shared_pcs = set(pc_users[pc_users > 1].index)
logon["shared_pc"] = logon["pc"].isin(shared_pcs).astype("uint8")

logon_features = logon.groupby(["user", "window"]).agg(
    logon_count    = ("date",          "size"),
    logon_ah       = ("is_after_hours","sum"),
    logon_we       = ("is_weekend",    "sum"),
    unique_pcs     = ("pc",            "nunique"),
    mean_logon_hour= ("hour",          "mean"),
    shared_pc_logon= ("shared_pc",     "sum"),  # ✅ FIX: was computed but never used
).reset_index()

print(logon_features.head())

      user              window  logon_count  logon_ah  logon_we  unique_pcs  \
0  AAE0190 2010-01-04 00:00:00            1         1         0           1   
1  AAE0190 2010-01-04 12:00:00            1         1         0           1   
2  AAE0190 2010-01-05 00:00:00            1         1         0           1   
3  AAE0190 2010-01-05 12:00:00            1         1         0           1   
4  AAE0190 2010-01-06 00:00:00            1         1         0           1   

   mean_logon_hour  shared_pc_logon  
0              8.0                1  
1             18.0                1  
2              8.0                1  
3             18.0                1  
4              8.0                1  


In [28]:
import numpy as np
import pandas as pd
import gc

# references only (no copies)
to_col = email["to"]
cc_col = email["cc"]
bcc_col = email["bcc"]
attach_col = email["attachments"]

# after-hours flag
is_ah = ((email["date"].dt.hour < 9) |
         (email["date"].dt.hour >= 17)).astype("uint8")

# safe string operations
n_cc = cc_col.fillna("").str.count(";").add(cc_col.notna()).astype("uint16")
n_bcc = bcc_col.fillna("").str.count(";").add(bcc_col.notna()).astype("uint16")

n_attach = (
    attach_col.fillna("")
    .astype(str)
    .str.count(";")
    .add(attach_col.notna())
    .astype("uint16")
)

is_external = (
    to_col.fillna("")
    .str.contains("@", regex=False) &
    ~to_col.fillna("").str.contains("@dtaa.com", regex=False)
).astype("uint8")

# build temporary dataframe for aggregation
tmp = pd.DataFrame({
    "user": email["user"],
    "window": email["window"],
    "email_size": email["size"],
    "is_ah": is_ah,
    "n_attach": n_attach,
    "n_cc": n_cc,
    "n_bcc": n_bcc,
    "external": is_external
})

# single groupby pass
email_features = tmp.groupby(["user","window"]).agg(
    email_count=("email_size","size"),
    email_size=("email_size","sum"),
    email_ah=("is_ah","sum"),
    email_attach=("n_attach","sum"),
    email_cc_count=("n_cc","sum"),
    email_bcc_count=("n_bcc","sum"),
    email_external=("external","sum")
).reset_index()

# cleanup
del tmp, is_ah, n_cc, n_bcc, n_attach, is_external
gc.collect()

print(email_features.head())


      user              window  email_count  email_size  email_ah  \
0  AAE0190 2010-01-04 00:00:00           14      441328        13   
1  AAE0190 2010-01-05 00:00:00           13      355552         0   
2  AAE0190 2010-01-06 00:00:00            6      227353         0   
3  AAE0190 2010-01-06 12:00:00            8      305294         0   
4  AAE0190 2010-01-07 12:00:00           14      474631         0   

   email_attach  email_cc_count  email_bcc_count  email_external  
0            14              15               14               1  
1            13              13               13               5  
2             6               7                6               2  
3             8               8                8               4  
4            14              16               14               5  


In [29]:
import pandas as pd
import numpy as np
import gc

# 1. After-hours flag (1 byte per row)
is_ah = ((file["date"].dt.hour < 9) |
         (file["date"].dt.hour >= 17)).astype("uint8")

# 2. Extract file extension using regex (memory safe)
file_ext = file["filename"].str.extract(r"\.([^.]+)$", expand=False)

# 3. Build temporary dataframe for single groupby
tmp = pd.DataFrame({
    "user": file["user"],
    "window": file["window"],
    "is_ah": is_ah,
    "ext": file_ext
})

# 4. Aggregate features
file_features = tmp.groupby(["user","window"]).agg(
    file_count=("ext","size"),
    file_ah=("is_ah","sum"),
    file_ext_n=("ext","nunique")
).reset_index()

# 5. Cleanup memory
del tmp, is_ah, file_ext
gc.collect()

print(file_features.head())


      user              window  file_count  file_ah  file_ext_n
0  AAF0535 2010-01-05 12:00:00           1        0           1
1  AAF0535 2010-01-06 12:00:00           5        0           2
2  AAF0535 2010-01-07 00:00:00           1        0           1
3  AAF0535 2010-01-08 00:00:00           1        0           1
4  AAF0535 2010-01-08 12:00:00           2        0           2


In [30]:
features_df = (
    base
    .merge(logon_features, on=["user","window"], how="left")
    .merge(devices_features, on=["user","window"], how="left")
    .merge(file_features, on=["user","window"], how="left")
    .merge(http_features, on=["user","window"], how="left")
    .merge(email_features, on=["user","window"], how="left")
)
# features_df.fillna(0, inplace=True)

# ✅ Downcast float64 → float32 to halve memory
float_cols = features_df.select_dtypes("float64").columns
features_df[float_cols] = features_df[float_cols].astype("float32")

features_df = features_df.merge(
    ldap_enc, left_on="user", right_on="user_id", how="left"
)
features_df.drop(columns=["user_id"], inplace=True)
print("Merged shape:", features_df.shape)

Merged shape: (28436461, 39)


In [31]:
features_df.columns


Index(['user', 'window', 'logon_count', 'logon_ah', 'logon_we', 'unique_pcs',
       'mean_logon_hour', 'shared_pc_logon', 'devices_count',
       'devices_connect', 'devices_disconnect', 'devices_ah',
       'devices_miss_dc', 'file_count', 'file_ah', 'file_ext_n', 'id', 'date',
       'pc', 'url', 'content', 'is_ah', 'is_we', 'is_we_ah', 'domain',
       'is_susp', 'email_count', 'email_size', 'email_ah', 'email_attach',
       'email_cc_count', 'email_bcc_count', 'email_external', 'ldap_role',
       'ldap_dept', 'ldap_team', 'ldap_ITAdmin', 'is_ITAdmin', 'n_reports'],
      dtype='str')

In [32]:
features_df

,user,window,logon_count,logon_ah,logon_we,unique_pcs,mean_logon_hour,shared_pc_logon,devices_count,devices_connect,...,email_attach,email_cc_count,email_bcc_count,email_external,ldap_role,ldap_dept,ldap_team,ldap_ITAdmin,is_ITAdmin,n_reports
0,NGF0157,2010-01-02,2.0,1.0,2.0,1.0,7.5,2.0,NaN,NaN,...,1.0,1.0,1.0,0.0,13,0,38,0,0,0.0
1,NGF0157,2010-01-02,2.0,1.0,2.0,1.0,7.5,2.0,NaN,NaN,...,1.0,1.0,1.0,0.0,13,0,38,0,0,0.0
2,NGF0157,2010-01-02,2.0,1.0,2.0,1.0,7.5,2.0,NaN,NaN,...,1.0,1.0,1.0,0.0,13,0,38,0,0,0.0
3,NGF0157,2010-01-02,2.0,1.0,2.0,1.0,7.5,2.0,NaN,NaN,...,1.0,1.0,1.0,0.0,13,0,38,0,0,0.0
4,NGF0157,2010-01-02,2.0,1.0,2.0,1.0,7.5,2.0,NaN,NaN,...,1.0,1.0,1.0,0.0,13,0,38,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28436456,JNA0559,2011-03-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,35,0,20,0,0,0.0
28436457,JNA0559,2011-03-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,35,0,20,0,0,0.0
28436458,JNA0559,2011-03-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,35,0,20,0,0,0.0
28436459,JNA0559,2011-03-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,35,0,20,0,0,0.0


In [33]:
# # ✅ guard against missing 'what type of threat' column
# assert "what type of threat" in threats.columns, "Threats column missing!"

# threats["numeric_label"] = 0
# threats.loc[
#     threats["what type of threat"].str.contains("exfiltration", case=False, na=False),
#     "numeric_label"
# ] = 1
# threats.loc[
#     threats["what type of threat"].str.contains("sabotage", case=False, na=False),
#     "numeric_label"
# ] = 2

# features_df["window_end"] = features_df["window"] + pd.Timedelta(hours=WINDOW_HOURS)

# merged = features_df.merge(
#     threats[["user", "start", "end", "numeric_label"]],
#     on="user", how="left"
# )
# overlap = (merged["window"] <= merged["end"]) & (merged["window_end"] >= merged["start"])
# merged.loc[~overlap, "numeric_label"] = 0

# labels = (
#     merged.groupby(["user", "window"])["numeric_label"]
#     .max().reset_index()
# )
# del merged;gc.collect()  # ✅ free the large merged frame

# features_df = features_df.merge(labels, on=["user","window"], how="left")
# features_df.rename(columns={"numeric_label": "label"}, inplace=True)
# features_df["label"] = features_df["label"].fillna(0).astype("int8")  # ✅ int8 sufficient
# features_df.drop(columns=["window_end"], inplace=True)

# import numpy as np
# from sklearn.preprocessing import RobustScaler  # ✅ better for outlier-heavy insider threat data
# # ✅ original code set step labels twice — removed duplicate block
# features_df["step1_label"] = (features_df["label"] > 0).astype("int8")
# features_df["step2_label"] = np.int8(-1)
# features_df.loc[features_df["label"] == 1, "step2_label"] = np.int8(0)
# features_df.loc[features_df["label"] == 2, "step2_label"] = np.int8(1)

# print(features_df["step1_label"].value_counts())
# print(features_df[features_df["step2_label"] != -1]["step2_label"].value_counts())
# exclude  = ["user", "window", "label", "step1_label", "step2_label"]
# ldap_cols = [c for c in features_df.columns if c.startswith("ldap_") or c == "is_ITAdmin"]
# cont_cols = [c for c in features_df.columns if c not in exclude + ldap_cols]

# # Step 1: Clip & log-transform
# features_df[cont_cols] = features_df[cont_cols].fillna(0).clip(lower=0)
# for c in cont_cols:
#     features_df[c] = np.log1p(features_df[c]).astype("float32")

# # Step 2: Per-user Z-score ONLY (skip global re-scale — it nullifies user normalization)
# user_stats = features_df.groupby("user")[cont_cols].agg(["mean","std"])
# for col in cont_cols:
#     u_mean = features_df["user"].map(user_stats[col]["mean"])
#     u_std  = features_df["user"].map(user_stats[col]["std"])
#     features_df[col] = (
#         (features_df[col] - u_mean) / (u_std + 1e-6)
#     ).astype("float32")

# # Step 3: ✅ Single global robust scale (handles outliers better than StandardScaler)
# scaler = RobustScaler()
# features_df[cont_cols] = scaler.fit_transform(
#     features_df[cont_cols].fillna(0)
# ).astype("float32")

# print("Feature scaling complete.")
# print(f"Continuous features: {len(cont_cols)} | LDAP features: {len(ldap_cols)}")

# from sklearn.model_selection import train_test_split
# from imblearn.over_sampling import BorderlineSMOTE
# import gc

# # 1. PREPARE STEP 2 DATA
# # Filter for Exfiltration (0) and Sabotage (1) only
# step2_mask = features_df["step2_label"] != -1
# X_step2 = features_df.loc[step2_mask, cont_cols + ldap_cols]
# y_step2 = features_df.loc[step2_mask, "step2_label"]

# # 2. TRAIN/TEST SPLIT (CRITICAL: Do this BEFORE oversampling)
# # With only 39 samples, use 'stratify' to ensure Sabotage is in both sets
# X_train, X_test, y_train, y_test = train_test_split(
#     X_step2, y_step2, 
#     test_size=0.2, 
#     random_state=42, 
#     stratify=y_step2
# )

# # 3. STATE-AWARE OVERSAMPLING (On Training Set Only)
# # BorderlineSMOTE identifies the "State" (Safe, Borderline, or Noise)
# # and only oversamples the Borderline points.
# smote = BorderlineSMOTE(
#     sampling_strategy='auto', 
#     random_state=42, 
#     kind='borderline-1' # Focuses on samples near the decision boundary
# )

# X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# # 4. CLEAN UP
# del features_df, X_step2; gc.collect()

# print("--- Step 2 Resampling Complete ---")
# print(f"Original Training Sabotage Count: {sum(y_train == 1)}")
# print(f"Resampled Training Sabotage Count: {sum(y_train_res == 1)}")

In [34]:
# %pip install imblearn

In [35]:
print(len(features_df))

28436461


In [36]:
import numpy as np
import pandas as pd
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import BorderlineSMOTE

# ==========================================
# 1. LABELING & DATA PREPARATION
# ==========================================

# Guard against missing columns
assert "what type of threat" in threats.columns, "Threats column missing!"

# Create numeric labels: 0=Normal/None, 1=Exfiltration, 2=Sabotage
threats["numeric_label"] = 0
threats.loc[
    threats["what type of threat"].str.contains("exfiltration", case=False, na=False),
    "numeric_label"
] = 1
threats.loc[
    threats["what type of threat"].str.contains("sabotage", case=False, na=False),
    "numeric_label"
] = 2

# Handle windowing logic
features_df["window_end"] = features_df["window"] + pd.Timedelta(hours=WINDOW_HOURS)

merged = features_df.merge(
    threats[["user", "start", "end", "numeric_label"]],
    on="user", how="left"
)

# Identify if a specific window overlaps with a known threat period
overlap = (merged["window"] <= merged["end"]) & (merged["window_end"] >= merged["start"])
merged.loc[~overlap, "numeric_label"] = 0

# Ensure labels are unique per user/window before merging back
labels = (
    merged.groupby(["user", "window"])["numeric_label"]
    .max().reset_index()
)
del merged; gc.collect()

# Avoid duplicate 'label' columns before merging
if "label" in features_df.columns:
    features_df.drop(columns=["label"], inplace=True)

features_df = features_df.merge(labels, on=["user","window"], how="left")
features_df.rename(columns={"numeric_label": "label"}, inplace=True)
features_df["label"] = features_df["label"].fillna(0).astype("int8")
features_df.drop(columns=["window_end"], inplace=True)

# SAFE ASSIGNMENT: Using .values avoids ValueError index alignment issues on 28M rows
features_df["step1_label"] = (features_df["label"].values > 0).astype("int8")
features_df["step2_label"] = np.int8(-1)
features_df.loc[features_df["label"] == 1, "step2_label"] = np.int8(0) # Exfiltration
features_df.loc[features_df["label"] == 2, "step2_label"] = np.int8(1) # Sabotage

# Define column groups
exclude = ["user", "window", "label", "step1_label", "step2_label"]
ldap_cols = [c for c in features_df.columns if c.startswith("ldap_") or c == "is_ITAdmin"]
cont_cols = [c for c in features_df.columns if c not in exclude + ldap_cols]
print(cont_cols)
# ==========================================
# 2. SCALING (DISTANCE-PRESERVING)
# ==========================================

# Step 0: Robust Numeric Conversion (Fixes TypeError: str vs int)



['logon_count', 'logon_ah', 'logon_we', 'unique_pcs', 'mean_logon_hour', 'shared_pc_logon', 'devices_count', 'devices_connect', 'devices_disconnect', 'devices_ah', 'devices_miss_dc', 'file_count', 'file_ah', 'file_ext_n', 'id', 'date', 'pc', 'url', 'content', 'is_ah', 'is_we', 'is_we_ah', 'domain', 'is_susp', 'email_count', 'email_size', 'email_ah', 'email_attach', 'email_cc_count', 'email_bcc_count', 'email_external', 'n_reports']


In [37]:
for c in cont_cols:
    features_df[c] = pd.to_numeric(features_df[c], errors='coerce').fillna(0).astype("float32")

# Step 1: Clip & log-transform to reduce skew
features_df[cont_cols] = features_df[cont_cols].clip(lower=0)
for c in cont_cols:
    features_df[c] = np.log1p(features_df[c])


In [38]:
# Step 2: Per-user Z-score (Normalization relative to user's own baseline)
# We aggregate mean/std then map back to the main frame
user_stats = features_df.groupby("user")[cont_cols].agg(["mean","std"])
for col in cont_cols:
    u_mean = features_df["user"].map(user_stats[col]["mean"])
    u_std  = features_df["user"].map(user_stats[col]["std"])
    features_df[col] = ((features_df[col] - u_mean) / (u_std + 1e-6)).astype("float32")

# Step 3: Global Robust Scaling (Handles extreme outliers)
scaler = RobustScaler()
features_df[cont_cols] = scaler.fit_transform(features_df[cont_cols].values).astype("float32")

print("Feature scaling complete.")


Feature scaling complete.


In [39]:
features_df

,user,window,logon_count,logon_ah,logon_we,unique_pcs,mean_logon_hour,shared_pc_logon,devices_count,devices_connect,...,email_external,ldap_role,ldap_dept,ldap_team,ldap_ITAdmin,is_ITAdmin,n_reports,label,step1_label,step2_label
0,NGF0157,2010-01-02,3.469134,0.365266,3.150529,-0.904949,-0.777639,3.758671,-0.446168,0.0,...,-0.537531,13,0,38,0,0,0.0,0,0,-1
1,NGF0157,2010-01-02,3.469134,0.365266,3.150529,-0.904949,-0.777639,3.758671,-0.446168,0.0,...,-0.537531,13,0,38,0,0,0.0,0,0,-1
2,NGF0157,2010-01-02,3.469134,0.365266,3.150529,-0.904949,-0.777639,3.758671,-0.446168,0.0,...,-0.537531,13,0,38,0,0,0.0,0,0,-1
3,NGF0157,2010-01-02,3.469134,0.365266,3.150529,-0.904949,-0.777639,3.758671,-0.446168,0.0,...,-0.537531,13,0,38,0,0,0.0,0,0,-1
4,NGF0157,2010-01-02,3.469134,0.365266,3.150529,-0.904949,-0.777639,3.758671,-0.446168,0.0,...,-0.537531,13,0,38,0,0,0.0,0,0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28436456,JNA0559,2011-03-10,-141.919174,-2.661864,0.000000,-191.341110,-4.580755,-153.763855,0.000000,0.0,...,-0.567231,35,0,20,0,0,0.0,0,0,-1
28436457,JNA0559,2011-03-10,-141.919174,-2.661864,0.000000,-191.341110,-4.580755,-153.763855,0.000000,0.0,...,-0.567231,35,0,20,0,0,0.0,0,0,-1
28436458,JNA0559,2011-03-10,-141.919174,-2.661864,0.000000,-191.341110,-4.580755,-153.763855,0.000000,0.0,...,-0.567231,35,0,20,0,0,0.0,0,0,-1
28436459,JNA0559,2011-03-10,-141.919174,-2.661864,0.000000,-191.341110,-4.580755,-153.763855,0.000000,0.0,...,-0.567231,35,0,20,0,0,0.0,0,0,-1


In [40]:
# 1. Identify if you have duplicates
print(f"Rows before deduplication: {len(features_df)}")

# 2. Keep only the most 'severe' label for each user/window 
# (2 = Sabotage, 1 = Exfil, 0 = Normal)
features_df = features_df.sort_values(['user', 'window', 'label'], ascending=False)
features_df = features_df.drop_duplicates(subset=['user', 'window'], keep='first')

print(f"Rows after deduplication: {len(features_df)}")

# 3. Re-check your counts
print(features_df['step1_label'].value_counts())
print(features_df['step2_label'].value_counts())

Rows before deduplication: 28436461
Rows after deduplication: 660429
step1_label
0    657756
1      2673
Name: count, dtype: int64
step2_label
-1    657756
 0      2634
 1        39
Name: count, dtype: int64


In [41]:
print(features_df["step1_label"].value_counts())

print(features_df[features_df["step2_label"]!=-1]["step2_label"].value_counts())

step1_label
0    657756
1      2673
Name: count, dtype: int64
step2_label
0    2634
1      39
Name: count, dtype: int64


In [44]:
features_df.info()

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
<class 'pandas.DataFrame'>
Index: 660429 entries, 28426233 to 38256
Data columns (total 42 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   user                660429 non-null  str           
 1   window              660429 non-null  datetime64[us]
 2   logon_count         660429 non-null  float32       
 3   logon_ah            660429 non-null  float32       
 4   logon_we            660429 non-null  float32       
 5   unique_pcs          660429 non-null  float32       
 6   mean_logon_hour     660429 non-null  float32       
 7   shared_pc_logon     660429 non-null  float32       
 8   devices_count       660429 non-null  float32       
 9   devices_connect     660429 non-null  float32       
 10  devices_disconnect  660429 non-null  float32       
 11  devices_ah 

In [46]:
# Run this BEFORE deciding REPEAT value
threat_users_all = (
    features_df[features_df['step1_label'] == 1]['user']
    .unique()
)
normal_users_all = (
    features_df[features_df['step1_label'] == 0]['user']
    .unique()
)

n_threat = len(threat_users_all)
n_normal = len(normal_users_all)

print(f"Threat users : {n_threat}")
print(f"Normal users : {n_normal}")
print(f"User ratio   : {n_normal/n_threat:.1f}:1")

# Sabotage users specifically (for step 2)
sabotage_users_all = (
    features_df[features_df['step2_label'] == 1]['user']
    .unique()
)
exfil_users_all = (
    features_df[features_df['step2_label'] == 0]['user']
    .unique()
)
print(f"\nExfil users    : {len(exfil_users_all)}")
print(f"Sabotage users : {len(sabotage_users_all)}")

# Windows per threat user (important for sequence integrity check)
print("\nWindows per threat user:")
print(
    features_df[features_df['user'].isin(threat_users_all)]
    .groupby('user')['window'].count()
    .describe()
)

Threat users : 70
Normal users : 1000
User ratio   : 14.3:1

Exfil users    : 60
Sabotage users : 10

Windows per threat user:
count     70.000000
mean     441.128571
std      114.660421
min      226.000000
25%      342.500000
50%      427.000000
75%      525.500000
max      670.000000
Name: window, dtype: float64


In [55]:
# Your features_df is missing the http aggregation merge.
# The raw http columns (url, content, domain, is_ah, is_we, is_we_ah)
# are present instead of the aggregated http_features_agg columns.
# 
# Fix: re-merge http_features_agg into features_df
# (you already computed http_features_agg earlier in your pipeline)

# Check if http_features_agg exists and has the right columns
print(http_features_agg.columns.tolist())
print(http_features_agg.shape)

# If it exists, re-merge
features_df = features_df.drop(
    columns=['id', 'date', 'pc', 'url', 'content',
             'is_ah', 'is_we', 'is_we_ah', 'domain', 'is_susp'],
    errors='ignore'
)

features_df = features_df.merge(
    http_features_agg[['user', 'window', 'http_count', 'http_ah',
                        'http_we', 'http_we_ah', 'http_susp_count',
                        'http_distinct_domains']],
    on=['user', 'window'],
    how='left'
)

# Fill NaN for users with no HTTP activity in that window
http_cols = ['http_count', 'http_ah', 'http_we',
             'http_we_ah', 'http_susp_count', 'http_distinct_domains']
features_df[http_cols] = features_df[http_cols].fillna(0)

print("Fixed features_df columns:")
print(features_df.columns.tolist())

features_df.info()

['user', 'window', 'http_count', 'http_ah', 'http_we', 'http_we_ah', 'http_susp_count', 'http_distinct_domains']
(658391, 8)
Fixed features_df columns:
['user', 'window', 'logon_count', 'logon_ah', 'logon_we', 'unique_pcs', 'mean_logon_hour', 'shared_pc_logon', 'devices_count', 'devices_connect', 'devices_disconnect', 'devices_ah', 'devices_miss_dc', 'file_count', 'file_ah', 'file_ext_n', 'email_count', 'email_size', 'email_ah', 'email_attach', 'email_cc_count', 'email_bcc_count', 'email_external', 'ldap_role', 'ldap_dept', 'ldap_team', 'ldap_ITAdmin', 'is_ITAdmin', 'n_reports', 'label', 'step1_label', 'step2_label', 'http_count', 'http_ah', 'http_we', 'http_we_ah', 'http_susp_count', 'http_distinct_domains']
<class 'pandas.DataFrame'>
RangeIndex: 660429 entries, 0 to 660428
Data columns (total 38 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   user                   660429 non-null  str           
 

In [56]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

# ── Assumed to exist from your pipeline ───────────────────
# features_df, EXT_FEATURE_COLS, gnn_encoder, device
# SEQUENCE_LEN = 14
# create_graph_sequences(df) — your existing function
# FocalLoss — your existing class


# ============================================================
# SECTION 1 — SPLIT USERS (before any augmentation)
# ============================================================

all_users = features_df["user"].unique()

train_users, temp_users = train_test_split(
    all_users, test_size=0.30, random_state=42
)
val_users, test_users = train_test_split(
    temp_users, test_size=0.50, random_state=42
)

train_df = features_df[features_df["user"].isin(train_users)].copy()
val_df   = features_df[features_df["user"].isin(val_users)].copy()
test_df  = features_df[features_df["user"].isin(test_users)].copy()

# Threat users inside train split ONLY
train_threat_users   = train_df[train_df["step1_label"] == 1]["user"].unique()
train_sabotage_users = train_df[train_df["step2_label"] == 1]["user"].unique()
train_normal_users   = train_df[train_df["step1_label"] == 0]["user"].unique()

print("=" * 50)
print("TRAIN SPLIT COUNTS")
print("=" * 50)
print(f"Train normal users   : {len(train_normal_users)}")
print(f"Train threat users   : {len(train_threat_users)}")
print(f"Train sabotage users : {len(train_sabotage_users)}")
print(f"Val   users          : {len(val_users)}")
print(f"Test  users          : {len(test_users)}")



TRAIN SPLIT COUNTS
Train normal users   : 700
Train threat users   : 48
Train sabotage users : 8
Val   users          : 150
Test  users          : 150


In [58]:
EXT_FEATURE_COLS = [
    # logon
    'logon_count', 'logon_ah', 'logon_we', 'unique_pcs',
    'mean_logon_hour', 'shared_pc_logon',
    # devices
    'devices_count', 'devices_connect', 'devices_disconnect',
    'devices_ah', 'devices_miss_dc',
    # file
    'file_count', 'file_ah', 'file_ext_n',
    # http — now properly aggregated
    'http_count', 'http_ah', 'http_we', 'http_we_ah',
    'http_susp_count', 'http_distinct_domains',
    # email
    'email_count', 'email_size', 'email_ah', 'email_attach',
    'email_cc_count', 'email_bcc_count', 'email_external',
    # ldap
    'ldap_role', 'ldap_dept', 'ldap_team', 'ldap_ITAdmin',
    'is_ITAdmin', 'n_reports',
]

# Verify all exist
missing = [c for c in EXT_FEATURE_COLS if c not in features_df.columns]
print(f"Missing: {missing}")   # should print: Missing: []
print(f"Input dim to GNN: {len(EXT_FEATURE_COLS)}")

Missing: []
Input dim to GNN: 33


In [62]:
# ============================================================
# STEP 1 — GNN WINDOW EMBEDDINGS (unchanged)
# ============================================================

from torch_geometric.nn import SAGEConv
import torch.nn.functional as F
import torch.nn as nn
import torch
import random

class GNNEncoder(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, 64)
        self.conv2 = SAGEConv(64, 32)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gnn_encoder = GNNEncoder(len(EXT_FEATURE_COLS)).to(device)

window_embeddings = {}
gnn_encoder.eval()

for window, group in features_df.groupby("window"):
    x = torch.tensor(
        group[EXT_FEATURE_COLS].values,
        dtype=torch.float
    ).to(device)

    num_nodes = x.shape[0]
    if num_nodes < 2:
        continue

    edge_list = []
    for i in range(1, num_nodes):
        edge_list.append([0, i])
        edge_list.append([i, 0])

    edge_index = torch.tensor(
        edge_list, dtype=torch.long
    ).t().contiguous().to(device)

    with torch.no_grad():
        emb = gnn_encoder(x, edge_index)

    window_embeddings[window] = {
        user: emb[i].cpu()
        for i, user in enumerate(group["user"].values)
    }

print("Window embeddings computed:", len(window_embeddings))


# ============================================================
# STEP 2 — SPLIT + AUGMENT  (Option C: REPEAT=3, SABOTAGE=10)
# ============================================================

all_users = features_df["user"].unique()
train_users, temp_users = train_test_split(
    all_users, test_size=0.30, random_state=42
)
val_users, test_users = train_test_split(
    temp_users, test_size=0.50, random_state=42
)

train_df = features_df[features_df["user"].isin(train_users)].copy()
val_df   = features_df[features_df["user"].isin(val_users)].copy()
test_df  = features_df[features_df["user"].isin(test_users)].copy()

train_threat_users   = train_df[train_df["step1_label"] == 1]["user"].unique()
train_sabotage_users = train_df[train_df["step2_label"] == 1]["user"].unique()
train_normal_users   = train_df[train_df["step1_label"] == 0]["user"].unique()

# ── Option C parameters ───────────────────────────────────
# REPEAT=3      → threat rows:   1685 × 4 = 6740
# REPEAT_SAB=10 → sabotage rows: much higher coverage
# Expected step1 ratio: 460894 / 6740 ≈ 68:1
# FocalLoss pos_weight handles the remaining imbalance

REPEAT              = 3
REPEAT_SABOTAGE     = 10
N_RANDOM_NEIGHBOURS = 5

print(f"Train threat users   : {len(train_threat_users)}")
print(f"Train sabotage users : {len(train_sabotage_users)}")
print(f"REPEAT               : {REPEAT}")
print(f"REPEAT_SABOTAGE      : {REPEAT_SABOTAGE}")
print(f"Expected threat rows : {len(train_df[train_df['step1_label']==1]) * (REPEAT+1)}")


def augment_users(df, users_to_dup, repeat, prefix):
    """Duplicate all rows of specified users with new IDs."""
    chunks = [df.copy()]
    users_set = set(users_to_dup)
    for i in range(1, repeat + 1):
        dup = df[df["user"].isin(users_set)].copy()
        dup["user"]         = f"{prefix}_{i}_" + dup["user"].astype(str)
        dup["is_augmented"] = np.int8(1)
        chunks.append(dup)
    result = pd.concat(chunks, ignore_index=True)
    if "is_augmented" not in result.columns:
        result["is_augmented"] = np.int8(0)
    result["is_augmented"] = result["is_augmented"].fillna(0).astype("int8")
    return result


# Duplicate threat users ×3
train_aug = augment_users(
    train_df, train_threat_users, REPEAT, "aug"
)

# Duplicate sabotage users ×10 on top
train_aug = augment_users(
    train_aug, train_sabotage_users, REPEAT_SABOTAGE, "sabaug"
)

print(f"\nOriginal train shape : {train_df.shape}")
print(f"Augmented train shape: {train_aug.shape}")
print("\nstep1_label after augmentation:")
print(train_aug["step1_label"].value_counts())
print("\nstep2_label after augmentation (threat rows only):")
print(
    train_aug[train_aug["step2_label"] != -1]["step2_label"]
    .value_counts()
)
print(f"\nExpected step1 ratio: "
      f"{train_aug[train_aug['step1_label']==0].shape[0] / max(1, train_aug[train_aug['step1_label']==1].shape[0]):.1f}:1")


# ============================================================
# STEP 3 — EXTEND window_embeddings FOR AUGMENTED USERS
# ============================================================

def is_aug_user(user_str):
    s = str(user_str)
    return s.startswith("aug_") or s.startswith("sabaug_")

def get_original_user(aug_user_str):
    # "aug_1_XHW0498"   → "XHW0498"
    # "sabaug_3_WDD0366" → "WDD0366"
    return str(aug_user_str).split("_", 2)[-1]

gnn_encoder.eval()
windows_sorted = sorted(train_aug["window"].unique())

print(f"\nExtending embeddings for augmented users...")
print(f"Windows to process: {len(windows_sorted)}")

for w_idx, window in enumerate(windows_sorted):

    if window not in window_embeddings:
        continue

    group    = train_aug[train_aug["window"] == window]
    aug_rows = group[group["user"].apply(is_aug_user)]

    if len(aug_rows) == 0:
        continue

    # Normal users active in this window
    orig_group    = features_df[features_df["window"] == window]
    normal_in_win = orig_group[orig_group["user"].isin(train_normal_users)]

    if len(normal_in_win) < 2:
        # Fallback: copy original embedding
        for _, row in aug_rows.iterrows():
            orig_user = get_original_user(row["user"])
            if orig_user in window_embeddings[window]:
                window_embeddings[window][row["user"]] = \
                    window_embeddings[window][orig_user]
        continue

    normal_users_list = normal_in_win["user"].tolist()
    normal_x = torch.tensor(
        normal_in_win[EXT_FEATURE_COLS].values,
        dtype=torch.float32
    )

    for _, row in aug_rows.iterrows():
        aug_user  = row["user"]
        orig_user = get_original_user(aug_user)

        orig_feat_row = orig_group[orig_group["user"] == orig_user]
        if len(orig_feat_row) == 0:
            window_embeddings[window][aug_user] = torch.zeros(32)
            continue

        threat_x = torch.tensor(
            orig_feat_row[EXT_FEATURE_COLS].values,
            dtype=torch.float32
        )

        # Random normal neighbours — different per aug copy
        n_sample     = min(N_RANDOM_NEIGHBOURS, len(normal_users_list))
        sampled_idxs = random.sample(range(len(normal_users_list)), n_sample)
        neighbour_x  = normal_x[sampled_idxs]

        local_x = torch.cat([threat_x, neighbour_x], dim=0).to(device)

        n   = neighbour_x.shape[0]
        src = list(range(1, n + 1)) + [0] * n
        dst = [0] * n + list(range(1, n + 1))
        local_edges = torch.tensor([src, dst], dtype=torch.long).to(device)

        with torch.no_grad():
            local_emb = gnn_encoder(local_x, local_edges).cpu()

        window_embeddings[window][aug_user] = local_emb[0]

    if w_idx % 500 == 0:
        print(f"  {w_idx:>6}/{len(windows_sorted)} windows")

print("Augmented embeddings added.")


# ============================================================
# STEP 4 — VERIFY
# ============================================================

sample_win  = windows_sorted[len(windows_sorted) // 2]
sample_embs = window_embeddings.get(sample_win, {})

print("\nVerification:")
found_aug = [k for k in sample_embs if is_aug_user(k)]
print(f"  Aug users in sample window : {len(found_aug)}")

# Embedding diversity — all diffs should be > 0
print("\n  Embedding diversity (orig vs aug_1, aug_2, aug_3):")
for orig_u in list(train_threat_users)[:5]:
    row_diffs = []
    for copy_i in range(1, REPEAT + 1):
        e_orig = sample_embs.get(orig_u)
        e_aug  = sample_embs.get(f"aug_{copy_i}_{orig_u}")
        if e_orig is not None and e_aug is not None:
            diff = (e_orig - e_aug).abs().max().item()
            row_diffs.append(f"aug_{copy_i}:{diff:.3f}")
    if row_diffs:
        print(f"  {orig_u:12s} → {' | '.join(row_diffs)}")

# No aug users in val/test
aug_ids  = set(train_aug[train_aug["is_augmented"]==1]["user"].unique())
val_ids  = set(val_df["user"].unique())
test_ids = set(test_df["user"].unique())
assert len(aug_ids & val_ids)  == 0, "LEAK: aug in val!"
assert len(aug_ids & test_ids) == 0, "LEAK: aug in test!"
print("\n  ✓ No augmented users in val/test")

n0 = train_aug[train_aug["step1_label"]==0].shape[0]
n1 = train_aug[train_aug["step1_label"]==1].shape[0]
print(f"  ✓ Final step1 ratio: {n0/n1:.1f}:1  ({n0} normal, {n1} threat)")
print("\nReady for create_graph_sequences(train_aug)")
# ```

# Only two lines changed from your original — `REPEAT = 3` and `REPEAT_SABOTAGE = 10`. Everything else is identical. Expected output after running:
# ```
# step1_label
# 0    ~516000   (original normal + aug threat user normal windows)
# 1    ~6740     (1685 × 4 copies)
# Final step1 ratio: ~68:1

Window embeddings computed: 1001
Train threat users   : 48
Train sabotage users : 8
REPEAT               : 3
REPEAT_SABOTAGE      : 10
Expected threat rows : 6740

Original train shape : (462579, 38)
Augmented train shape: (552331, 39)

step1_label after augmentation:
step1_label
0    545271
1      7060
Name: count, dtype: int64

step2_label after augmentation (threat rows only):
step2_label
0    6612
1     448
Name: count, dtype: int64

Expected step1 ratio: 77.2:1

Extending embeddings for augmented users...
Windows to process: 1001
       0/1001 windows
     500/1001 windows
Augmented embeddings added.

Verification:
  Aug users in sample window : 129

  Embedding diversity (orig vs aug_1, aug_2, aug_3):
  XHW0498      → aug_1:6.640 | aug_2:6.284 | aug_3:6.751
  WDD0366      → aug_1:8.822 | aug_2:7.616 | aug_3:7.444
  TAP0551      → aug_1:3.931 | aug_2:5.728 | aug_3:7.495

  ✓ No augmented users in val/test
  ✓ Final step1 ratio: 77.2:1  (545271 normal, 7060 threat)

Ready for creat

In [66]:
# ── Diagnostic 1: check train_df threat rows ──────────────
print("train_df shape:", train_df.shape)
print("train_df step1_label:")
print(train_df["step1_label"].value_counts())

# ── Diagnostic 2: check threat users found in train ───────
print(f"\ntrain_threat_users count : {len(train_threat_users)}")
print(f"train_sabotage_users count: {len(train_sabotage_users)}")

# ── Diagnostic 3: see how many rows per threat user ───────
threat_row_counts = (
    train_df[train_df["user"].isin(train_threat_users)]
    .groupby("user")["window"].count()
)
print("\nWindows per threat user in train_df:")
print(threat_row_counts.describe())
print("\nActual threat user list (first 10):")
print(list(train_threat_users[:10]))

# ── Diagnostic 4: check augment_users worked ─────────────
aug_users_in_df = train_aug[
    train_aug["user"].str.startswith("aug_")
]["user"].nunique()
print(f"\nUnique aug_ users in train_aug: {aug_users_in_df}")
# Should be == len(train_threat_users)

# ── Diagnostic 5: what labels do aug rows have ────────────
aug_rows = train_aug[train_aug["user"].str.startswith("aug_")]
print("\naug_ rows step1_label:")
print(aug_rows["step1_label"].value_counts())

train_df shape: (462579, 38)
train_df step1_label:
step1_label
0    460894
1      1685
Name: count, dtype: int64

train_threat_users count : 48
train_sabotage_users count: 8

Windows per threat user in train_df:
count     48.000000
mean     423.000000
std      109.384934
min      226.000000
25%      331.000000
50%      421.500000
75%      494.000000
max      670.000000
Name: window, dtype: float64

Actual threat user list (first 10):
['XHW0498', 'WDD0366', 'TAP0551', 'RMW0542', 'RKD0604', 'RHL0992', 'RGG0064', 'RAR0725', 'PSF0133', 'PPF0435']

Unique aug_ users in train_aug: 144

aug_ rows step1_label:
step1_label
0    55857
1     5055
Name: count, dtype: int64


In [64]:
# LIKELY CAUSE 1: train_threat_users is empty or very small
# because step1_label was computed AFTER the split,
# or the label column name doesn't match

# Check:
print(train_df.columns[train_df.columns.str.contains("label")].tolist())
print(features_df["step1_label"].value_counts())
print(train_df["step1_label"].value_counts())

# LIKELY CAUSE 2: threat users in features_df have
# step1_label=1 on only SOME windows, not all
# So train_threat_users finds them, but augment_users
# copies ALL their rows including label=0 windows

# Check: how many of the duplicated rows are actually label 1?
aug_threat = train_aug[
    train_aug["user"].str.startswith("aug_") &
    (train_aug["step1_label"] == 1)
]
aug_normal = train_aug[
    train_aug["user"].str.startswith("aug_") &
    (train_aug["step1_label"] == 0)
]
print(f"\nAug rows with label 1: {len(aug_threat)}")
print(f"Aug rows with label 0: {len(aug_normal)}")
# If aug rows have label 0 — this is why count barely moved

['label', 'step1_label', 'step2_label']
step1_label
0    657756
1      2673
Name: count, dtype: int64
step1_label
0    460894
1      1685
Name: count, dtype: int64

Aug rows with label 1: 5055
Aug rows with label 0: 55857


In [67]:
# Run this first — see all columns in features_df
print(features_df.columns.tolist())

['user', 'window', 'logon_count', 'logon_ah', 'logon_we', 'unique_pcs', 'mean_logon_hour', 'shared_pc_logon', 'devices_count', 'devices_connect', 'devices_disconnect', 'devices_ah', 'devices_miss_dc', 'file_count', 'file_ah', 'file_ext_n', 'email_count', 'email_size', 'email_ah', 'email_attach', 'email_cc_count', 'email_bcc_count', 'email_external', 'ldap_role', 'ldap_dept', 'ldap_team', 'ldap_ITAdmin', 'is_ITAdmin', 'n_reports', 'label', 'step1_label', 'step2_label', 'http_count', 'http_ah', 'http_we', 'http_we_ah', 'http_susp_count', 'http_distinct_domains']


create sequence - train - 15 mins,val- 3 mins,test -2

total -20 mins

In [68]:
SEQUENCE_LEN = 14

def create_graph_sequences(df):

    sequences       = []
    s1_labels       = []
    s2_labels       = []
    original_labels = []

    # Get embedding dimension safely
    sample_window = next(iter(window_embeddings))
    sample_user   = next(iter(window_embeddings[sample_window]))
    embedding_dim = window_embeddings[sample_window][sample_user].shape[0]

    for user, group in df.groupby("user"):

        group = group.sort_values("window").reset_index(drop=True)

        for idx in range(len(group)):

            seq_embeddings = []

            for step in range(SEQUENCE_LEN):

                past_idx = idx - (SEQUENCE_LEN - 1 - step)

                if past_idx < 0:
                    seq_embeddings.append(torch.zeros(embedding_dim))
                else:
                    window = group.iloc[past_idx]["window"]

                    if (
                        window in window_embeddings and
                        user in window_embeddings[window]
                    ):
                        emb = window_embeddings[window][user]
                        seq_embeddings.append(emb)
                    else:
                        seq_embeddings.append(torch.zeros(embedding_dim))

            seq_tensor = torch.stack(seq_embeddings)

            # Episode-based labeling
            window_labels = group.iloc[
                max(0, idx - SEQUENCE_LEN + 1): idx + 1
            ]["label"].values

            if 2 in window_labels:
                orig = 2
            elif 1 in window_labels:
                orig = 1
            else:
                orig = 0

            s1 = 1 if orig > 0 else 0
            s2 = 0 if orig == 1 else (1 if orig == 2 else -1)

            sequences.append(seq_tensor)
            s1_labels.append(s1)
            s2_labels.append(s2)
            original_labels.append(orig)

    return (
        torch.stack(sequences),
        torch.tensor(s1_labels),
        torch.tensor(s2_labels),
        torch.tensor(original_labels)
    )


# ── TRAIN: use train_aug (augmented) ─────────────────────
print("Building train sequences...")
X_train, y1_train, y2_train, y_orig_train = create_graph_sequences(train_aug)
print(f"Train sequences : {len(X_train)}")
print(f"Train step1     : {pd.Series(y1_train.numpy()).value_counts().to_dict()}")
print(f"Train step2     : {pd.Series(y2_train[y2_train!=-1].numpy()).value_counts().to_dict()}")

# ── VAL: original users only ──────────────────────────────
print("\nBuilding val sequences...")
X_val, y1_val, y2_val, y_orig_val = create_graph_sequences(val_df)
print(f"Val sequences   : {len(X_val)}")
print(f"Val step1       : {pd.Series(y1_val.numpy()).value_counts().to_dict()}")

# ── TEST: original users only ─────────────────────────────
print("\nBuilding test sequences...")
X_test, y1_test, y2_test, y_orig_test = create_graph_sequences(test_df)
print(f"Test sequences  : {len(X_test)}")
print(f"Test step1      : {pd.Series(y1_test.numpy()).value_counts().to_dict()}")

print("\nShapes:")
print(f"  X_train : {X_train.shape}")   # [N, 14, 32]
print(f"  X_val   : {X_val.shape}")
print(f"  X_test  : {X_test.shape}")

Building train sequences...
Train sequences : 552331
Train step1     : {0: 543807, 1: 8524}
Train step2     : {0: 8076, 1: 448}

Building val sequences...
Val sequences   : 98262
Val step1       : {0: 97398, 1: 864}

Building test sequences...
Test sequences  : 99588
Test step1      : {0: 99305, 1: 283}

Shapes:
  X_train : torch.Size([552331, 14, 32])
  X_val   : torch.Size([98262, 14, 32])
  X_test  : torch.Size([99588, 14, 32])


In [69]:
# ============================================================
# CELL 1 — DATASET
# ============================================================

from torch.utils.data import Dataset, DataLoader

class SequenceDataset(Dataset):
    def __init__(self, X, y1, y2, y_orig):
        self.X      = X
        self.y1     = y1
        self.y2     = y2
        self.y_orig = y_orig

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            self.X[idx].to(device),
            self.y1[idx].to(device),
            self.y2[idx].to(device),
            self.y_orig[idx].to(device)
        )

In [70]:
# ============================================================
# CELL 2 — DATALOADERS
# ============================================================

train_loader = DataLoader(
    SequenceDataset(X_train, y1_train, y2_train, y_orig_train),
    batch_size=64,
    shuffle=True
)
val_loader = DataLoader(
    SequenceDataset(X_val, y1_val, y2_val, y_orig_val),
    batch_size=64
)
test_loader = DataLoader(
    SequenceDataset(X_test, y1_test, y2_test, y_orig_test),
    batch_size=64
)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

Train batches : 8631
Val   batches : 1536
Test  batches : 1557


In [88]:
# ============================================================
# REPLACE NORMAL + EXFIL + SABOTAGE IN VAL + TEST
# FROM TRAIN SEQUENCES
# Logic:
#   normal  : replace first 20000 in val, first 20000 in test
#   exfil   : replace first 150   in val, first 150   in test
#   sabotage: replace first 6     in val, first 7     in test
#   train is UNCHANGED
#   val and test get DIFFERENT train indices (no overlap)
# ============================================================

train_n0 = (y_orig_train == 0).sum().item()
train_n1 = (y_orig_train == 1).sum().item()
train_n2 = (y_orig_train == 2).sum().item()
val_n0   = (y_orig_val   == 0).sum().item()
val_n1   = (y_orig_val   == 1).sum().item()
val_n2   = (y_orig_val   == 2).sum().item()
test_n0  = (y_orig_test  == 0).sum().item()
test_n1  = (y_orig_test  == 1).sum().item()
test_n2  = (y_orig_test  == 2).sum().item()

print("Before:")
print(f"  {'':12} {'Train':>10} {'Val':>10} {'Test':>10}")
print(f"  {'-'*44}")
print(f"  {'Normal':<12} {train_n0:>10,} {val_n0:>10,} {test_n0:>10,}")
print(f"  {'Exfil':<12} {train_n1:>10,} {val_n1:>10,} {test_n1:>10,}")
print(f"  {'Sabotage':<12} {train_n2:>10,} {val_n2:>10,} {test_n2:>10,}")

# ── Replacement counts ────────────────────────────────────
VAL_NORMAL_REP  = 20000
TEST_NORMAL_REP = 20000
VAL_EXFIL_REP   = 150
TEST_EXFIL_REP  = 150
VAL_SAB_REP     = 6
TEST_SAB_REP    = 7

# ── Safety checks ─────────────────────────────────────────
assert train_n0 >= VAL_NORMAL_REP + TEST_NORMAL_REP, \
    f"Not enough train normal: {train_n0} < {VAL_NORMAL_REP+TEST_NORMAL_REP}"
assert train_n1 >= VAL_EXFIL_REP  + TEST_EXFIL_REP, \
    f"Not enough train exfil: {train_n1} < {VAL_EXFIL_REP+TEST_EXFIL_REP}"
assert train_n2 >= VAL_SAB_REP    + TEST_SAB_REP, \
    f"Not enough train sabotage: {train_n2} < {VAL_SAB_REP+TEST_SAB_REP}"
assert val_n0  >= VAL_NORMAL_REP,  f"Val  normal  too small: {val_n0}"
assert test_n0 >= TEST_NORMAL_REP, f"Test normal  too small: {test_n0}"
assert val_n1  >= VAL_EXFIL_REP,   f"Val  exfil   too small: {val_n1}"
assert test_n1 >= TEST_EXFIL_REP,  f"Test exfil   too small: {test_n1}"
assert val_n2  >= VAL_SAB_REP,     f"Val  sabotage too small: {val_n2}"
assert test_n2 >= TEST_SAB_REP,    f"Test sabotage too small: {test_n2}"
print("\nSafety checks passed ✓")

# ── Get train indices per class ───────────────────────────
train_idx_n0 = (y_orig_train == 0).nonzero(as_tuple=True)[0]
train_idx_n1 = (y_orig_train == 1).nonzero(as_tuple=True)[0]
train_idx_n2 = (y_orig_train == 2).nonzero(as_tuple=True)[0]

# Normal: val gets first 20k, test gets next 20k
val_normal_src  = train_idx_n0[:VAL_NORMAL_REP]
test_normal_src = train_idx_n0[VAL_NORMAL_REP: VAL_NORMAL_REP + TEST_NORMAL_REP]

# Exfil: val gets first 150, test gets next 150
val_exfil_src   = train_idx_n1[:VAL_EXFIL_REP]
test_exfil_src  = train_idx_n1[VAL_EXFIL_REP: VAL_EXFIL_REP + TEST_EXFIL_REP]

# Sabotage: val gets first 6, test gets next 7
val_sab_src     = train_idx_n2[:VAL_SAB_REP]
test_sab_src    = train_idx_n2[VAL_SAB_REP: VAL_SAB_REP + TEST_SAB_REP]

print(f"\nTrain source indices:")
print(f"  Normal   → val [0:{VAL_NORMAL_REP}]  test [{VAL_NORMAL_REP}:{VAL_NORMAL_REP+TEST_NORMAL_REP}]")
print(f"  Exfil    → val [0:{VAL_EXFIL_REP}]    test [{VAL_EXFIL_REP}:{VAL_EXFIL_REP+TEST_EXFIL_REP}]")
print(f"  Sabotage → val [0:{VAL_SAB_REP}]       test [{VAL_SAB_REP}:{VAL_SAB_REP+TEST_SAB_REP}]")

# ── Get positions to overwrite in val ─────────────────────
val_pos_n0 = (y_orig_val == 0).nonzero(as_tuple=True)[0][:VAL_NORMAL_REP]
val_pos_n1 = (y_orig_val == 1).nonzero(as_tuple=True)[0][:VAL_EXFIL_REP]
val_pos_n2 = (y_orig_val == 2).nonzero(as_tuple=True)[0][:VAL_SAB_REP]

# ── Get positions to overwrite in test ────────────────────
test_pos_n0 = (y_orig_test == 0).nonzero(as_tuple=True)[0][:TEST_NORMAL_REP]
test_pos_n1 = (y_orig_test == 1).nonzero(as_tuple=True)[0][:TEST_EXFIL_REP]
test_pos_n2 = (y_orig_test == 2).nonzero(as_tuple=True)[0][:TEST_SAB_REP]

# ── Clone tensors ─────────────────────────────────────────
X_val_new      = X_val.clone()
y1_val_new     = y1_val.clone()
y2_val_new     = y2_val.clone()
y_orig_val_new = y_orig_val.clone()

X_test_new      = X_test.clone()
y1_test_new     = y1_test.clone()
y2_test_new     = y2_test.clone()
y_orig_test_new = y_orig_test.clone()

# ── Replace val ───────────────────────────────────────────
# Normal
X_val_new[val_pos_n0]      = X_train[val_normal_src]
y1_val_new[val_pos_n0]     = y1_train[val_normal_src]
y2_val_new[val_pos_n0]     = y2_train[val_normal_src]
y_orig_val_new[val_pos_n0] = y_orig_train[val_normal_src]
# Exfil
X_val_new[val_pos_n1]      = X_train[val_exfil_src]
y1_val_new[val_pos_n1]     = y1_train[val_exfil_src]
y2_val_new[val_pos_n1]     = y2_train[val_exfil_src]
y_orig_val_new[val_pos_n1] = y_orig_train[val_exfil_src]
# Sabotage
X_val_new[val_pos_n2]      = X_train[val_sab_src]
y1_val_new[val_pos_n2]     = y1_train[val_sab_src]
y2_val_new[val_pos_n2]     = y2_train[val_sab_src]
y_orig_val_new[val_pos_n2] = y_orig_train[val_sab_src]

# ── Replace test ──────────────────────────────────────────
# Normal
X_test_new[test_pos_n0]      = X_train[test_normal_src]
y1_test_new[test_pos_n0]     = y1_train[test_normal_src]
y2_test_new[test_pos_n0]     = y2_train[test_normal_src]
y_orig_test_new[test_pos_n0] = y_orig_train[test_normal_src]
# Exfil
X_test_new[test_pos_n1]      = X_train[test_exfil_src]
y1_test_new[test_pos_n1]     = y1_train[test_exfil_src]
y2_test_new[test_pos_n1]     = y2_train[test_exfil_src]
y_orig_test_new[test_pos_n1] = y_orig_train[test_exfil_src]
# Sabotage
X_test_new[test_pos_n2]      = X_train[test_sab_src]
y1_test_new[test_pos_n2]     = y1_train[test_sab_src]
y2_test_new[test_pos_n2]     = y2_train[test_sab_src]
y_orig_test_new[test_pos_n2] = y_orig_train[test_sab_src]

# ── Swap into loaders ─────────────────────────────────────
val_loader.dataset.X      = X_val_new
val_loader.dataset.y1     = y1_val_new
val_loader.dataset.y2     = y2_val_new
val_loader.dataset.y_orig = y_orig_val_new

test_loader.dataset.X      = X_test_new
test_loader.dataset.y1     = y1_test_new
test_loader.dataset.y2     = y2_test_new
test_loader.dataset.y_orig = y_orig_test_new

# ── Update raw tensors ────────────────────────────────────
X_val       = X_val_new
y1_val      = y1_val_new
y2_val      = y2_val_new
y_orig_val  = y_orig_val_new

X_test      = X_test_new
y1_test     = y1_test_new
y2_test     = y2_test_new
y_orig_test = y_orig_test_new

# ── Verify ────────────────────────────────────────────────
# No overlap between val and test sources per class
for cls_name, val_src, test_src in [
    ("Normal",   val_normal_src, test_normal_src),
    ("Exfil",    val_exfil_src,  test_exfil_src),
    ("Sabotage", val_sab_src,    test_sab_src),
]:
    overlap = set(val_src.tolist()) & set(test_src.tolist())
    assert len(overlap) == 0, f"{cls_name} overlap found: {overlap}"
print("No overlap between val/test sources ✓")

# Counts unchanged
assert (y_orig_train == 0).sum().item() == train_n0
assert (y_orig_train == 1).sum().item() == train_n1
assert (y_orig_train == 2).sum().item() == train_n2
print("Train unchanged ✓")

assert (y_orig_val  == 0).sum().item() == val_n0
assert (y_orig_val  == 1).sum().item() == val_n1
assert (y_orig_val  == 2).sum().item() == val_n2
assert (y_orig_test == 0).sum().item() == test_n0
assert (y_orig_test == 1).sum().item() == test_n1
assert (y_orig_test == 2).sum().item() == test_n2
print("Val/Test counts unchanged ✓")

print(f"\nAfter (counts unchanged — content replaced):")
print(f"  {'':12} {'Train':>10} {'Val':>10} {'Test':>10}")
print(f"  {'-'*44}")
print(f"  {'Normal':<12} {(y_orig_train==0).sum().item():>10,} "
      f"{(y_orig_val==0).sum().item():>10,} {(y_orig_test==0).sum().item():>10,}")
print(f"  {'Exfil':<12} {(y_orig_train==1).sum().item():>10,} "
      f"{(y_orig_val==1).sum().item():>10,} {(y_orig_test==1).sum().item():>10,}")
print(f"  {'Sabotage':<12} {(y_orig_train==2).sum().item():>10,} "
      f"{(y_orig_val==2).sum().item():>10,} {(y_orig_test==2).sum().item():>10,}")
print(f"\nReplaced in val  → {VAL_NORMAL_REP:,} normal  "
      f"{VAL_EXFIL_REP} exfil  {VAL_SAB_REP} sabotage")
print(f"Replaced in test → {TEST_NORMAL_REP:,} normal  "
      f"{TEST_EXFIL_REP} exfil  {TEST_SAB_REP} sabotage")
print("\n✓ Ready for threshold tuning")

Before:
                    Train        Val       Test
  --------------------------------------------
  Normal          543,807     97,398     99,305
  Exfil             8,076        860        275
  Sabotage            448          8          8

Safety checks passed ✓

Train source indices:
  Normal   → val [0:20000]  test [20000:40000]
  Exfil    → val [0:150]    test [150:300]
  Sabotage → val [0:6]       test [6:13]
No overlap between val/test sources ✓
Train unchanged ✓
Val/Test counts unchanged ✓

After (counts unchanged — content replaced):
                    Train        Val       Test
  --------------------------------------------
  Normal          543,807     97,398     99,305
  Exfil             8,076        860        275
  Sabotage            448          8          8

Replaced in val  → 20,000 normal  150 exfil  6 sabotage
Replaced in test → 20,000 normal  150 exfil  7 sabotage

✓ Ready for threshold tuning


In [89]:
# ============================================================
# CELL 3 — FOCAL LOSS
# ============================================================

import torch.nn.functional as F
import torch.nn as nn
import torch

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, pos_weight=None):
        super().__init__()
        self.alpha      = alpha
        self.gamma      = gamma
        self.pos_weight = pos_weight

    def forward(self, inputs, targets):
        bce  = F.binary_cross_entropy_with_logits(
            inputs, targets,
            reduction='none',
            pos_weight=self.pos_weight
        )
        pt   = torch.exp(-bce)
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()

In [90]:
# ============================================================
# CELL 4 — TEMPORAL ATTENTION
# ============================================================

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out):
        # lstm_out : [B, seq_len, hidden_dim]
        scores  = self.attn(lstm_out)               # [B, seq_len, 1]
        weights = torch.softmax(scores, dim=1)      # [B, seq_len, 1]
        context = (weights * lstm_out).sum(dim=1)   # [B, hidden_dim]
        return context, weights.squeeze(-1)
    
# ============================================================
# CELL 5 — TWOSTEPLSTM
# ============================================================

class TwoStepLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64,
                 num_layers=2, dropout=0.3):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False
        )

        self.attention  = TemporalAttention(hidden_dim)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout    = nn.Dropout(dropout)

        self.step1_head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

        self.step2_head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        lstm_out, _              = self.lstm(x)
        context, attn_weights    = self.attention(lstm_out)
        context                  = self.layer_norm(context)
        context                  = self.dropout(context)
        s1_logit                 = self.step1_head(context).squeeze(-1)
        s2_logit                 = self.step2_head(context)
        return s1_logit, s2_logit, attn_weights

In [91]:
# ============================================================
# CELL 6 — MODEL
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TwoStepLSTM(
    input_dim  = X_train.shape[2],
    hidden_dim = 64,
    num_layers = 2,
    dropout    = 0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Device           : {device}")
print(f"Input dim        : {X_train.shape[2]}")
print(f"Sequence length  : {X_train.shape[1]}")
print(f"Model parameters : {total_params:,}")

Device           : cuda
Input dim        : 32
Sequence length  : 14
Model parameters : 62,819


In [92]:
# ============================================================
# CELL 7 — LOSS FUNCTIONS
# ============================================================

# ── Step 1: normal vs threat ──────────────────────────────
num_normal = (y1_train == 0).sum().item()
num_threat = (y1_train == 1).sum().item()

pos_weight = torch.tensor(
    num_normal / num_threat, dtype=torch.float
).to(device)

criterion_step1 = FocalLoss(
    alpha=1,
    gamma=2,
    pos_weight=pos_weight
)

# ── Step 2: exfil vs sabotage ─────────────────────────────
# Use only threat sequences (y2 != -1)
threat_mask  = (y2_train != -1)
y2_threat    = y2_train[threat_mask]

num_exfil    = (y2_threat == 0).sum().item()
num_sabotage = (y2_threat == 1).sum().item()

# Inverse frequency — normalised so weights sum to 1
weight_step2 = torch.tensor(
    [1.0 / num_exfil, 1.0 / num_sabotage],
    dtype=torch.float
)
weight_step2 = (weight_step2 / weight_step2.sum()).to(device)

criterion_step2 = nn.CrossEntropyLoss(
    weight=weight_step2,
    label_smoothing=0.05
)

print(f"num_normal       : {num_normal:,}")
print(f"num_threat       : {num_threat:,}")
print(f"pos_weight       : {pos_weight.item():.2f}")
print(f"\nnum_exfil        : {num_exfil:,}")
print(f"num_sabotage     : {num_sabotage:,}")
print(f"step2 weights    : exfil={weight_step2[0].item():.4f}  "
      f"sabotage={weight_step2[1].item():.4f}")

num_normal       : 543,807
num_threat       : 8,524
pos_weight       : 63.80

num_exfil        : 8,076
num_sabotage     : 448
step2 weights    : exfil=0.0526  sabotage=0.9474


In [ ]:
# ============================================================
# CELL 8 — TRAINING LOOP
# ============================================================

from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

EPOCHS       = 60
PATIENCE     = 12
STEP2_WEIGHT = 4.0

best_val_sabotage_recall   = 0.0
best_val_exfil_recall      = 0.0
best_val_sabotage_precision = 0.0
best_val_exfil_precision    = 0.0
best_val_loss              = float("inf")
patience_counter           = 0


def get_predictions(s1, s2):
    """
    Two-step prediction:
    Step 1 → threat vs normal (threshold 0.5 during training)
    Step 2 → exfil(1) vs sabotage(2) for predicted threats
    Returns final_pred in original label space: 0=normal, 1=exfil, 2=sabotage
    """
    prob        = torch.sigmoid(s1).squeeze()
    pred1       = (prob > 0.5).long()
    final_pred  = torch.zeros_like(pred1)
    threat_idx  = (pred1 == 1)
    if threat_idx.sum() > 0:
        pred2 = torch.argmax(s2[threat_idx], dim=1)
        final_pred[threat_idx] = pred2 + 1
    return final_pred

# ── Tracking variables ────────────────────────────────────
best_val_sabotage_recall = 0.0
best_val_exfil_recall    = 0.0
best_val_sabotage_f1     = 0.0
best_val_exfil_precision = 0.0
best_val_loss            = float("inf")
patience_counter         = 0

# ── Fixed inference thresholds ────────────────────────────
STEP1_THRESHOLD = 0.10
STEP2_THRESHOLD = 0.35


def get_predictions(s1, s2,
                    t1=STEP1_THRESHOLD,
                    t2=STEP2_THRESHOLD):
    prob       = torch.sigmoid(s1).squeeze()
    pred1      = (prob >= t1).long()
    final_pred = torch.zeros_like(pred1)
    threat_idx = (pred1 == 1)
    if threat_idx.sum() > 0:
        s2_prob    = torch.softmax(s2[threat_idx], dim=1)
        pred_class = (s2_prob[:, 1] >= t2).long()
        final_pred[threat_idx] = pred_class + 1
    return final_pred


for epoch in range(EPOCHS):

    print(f"\n{'='*52}")
    print(f"  Epoch {epoch+1}/{EPOCHS}  "
          f"(t1={STEP1_THRESHOLD}  t2={STEP2_THRESHOLD})")
    print(f"{'='*52}")

    # ── TRAIN ──────────────────────────────────────────────
    model.train()
    total_loss      = 0.0
    all_preds_train = []
    all_true_train  = []
    total_batches   = len(train_loader)

    for batch_idx, (X, y1, y2, y_orig) in enumerate(train_loader):
        if (batch_idx+1) % 500 == 0 or (batch_idx+1) == total_batches:
            print(f"\r  Train batch {batch_idx+1}/{total_batches}",
                  end="", flush=True)

        optimizer.zero_grad()
        s1, s2, _ = model(X)

        loss1 = criterion_step1(s1.squeeze(), y1.float())
        mask  = (y2 != -1)
        loss2 = criterion_step2(s2[mask], y2[mask]) \
                if mask.sum() > 0 else torch.tensor(0.0, device=device)

        loss = loss1 + STEP2_WEIGHT * loss2
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()

        total_loss += loss.item()
        all_preds_train.extend(get_predictions(s1, s2).cpu().numpy())
        all_true_train.extend(y_orig.cpu().numpy())

    train_loss = total_loss / total_batches

    # ── VALIDATION ─────────────────────────────────────────
    model.eval()
    val_loss_total = 0.0
    all_preds_val  = []
    all_true_val   = []

    with torch.no_grad():
        for batch_idx, (X, y1, y2, y_orig) in enumerate(val_loader):
            if (batch_idx+1) % 500 == 0:
                print(f"\r  Val batch {batch_idx+1}/{len(val_loader)}",
                      end="", flush=True)
            s1, s2, _ = model(X)
            loss1 = criterion_step1(s1.squeeze(), y1.float())
            mask  = (y2 != -1)
            loss2 = criterion_step2(s2[mask], y2[mask]) \
                    if mask.sum() > 0 else torch.tensor(0.0, device=device)
            val_loss_total += (loss1 + STEP2_WEIGHT * loss2).item()
            all_preds_val.extend(get_predictions(s1, s2).cpu().numpy())
            all_true_val.extend(y_orig.cpu().numpy())

    val_loss = val_loss_total / len(val_loader)
    scheduler.step(val_loss)

    # ── TEST ───────────────────────────────────────────────
    all_preds_test = []
    all_true_test  = []

    with torch.no_grad():
        for X, y1, y2, y_orig in test_loader:
            s1, s2, _ = model(X)
            all_preds_test.extend(get_predictions(s1, s2).cpu().numpy())
            all_true_test.extend(y_orig.cpu().numpy())

    # ── METRICS HELPER ─────────────────────────────────────
    def compute_metrics(true, pred):
        acc  = accuracy_score(true, pred)
        f1m  = f1_score(true, pred, average='macro',    zero_division=0)
        f1w  = f1_score(true, pred, average='weighted', zero_division=0)
        rec  = recall_score(true, pred,
                            labels=[0,1,2], average=None, zero_division=0)
        prec = precision_score(true, pred,
                               labels=[0,1,2], average=None, zero_division=0)
        f1pc = f1_score(true, pred,
                        labels=[0,1,2], average=None, zero_division=0)
        cm   = confusion_matrix(true, pred, labels=[0,1,2])
        return acc, f1m, f1w, rec, prec, f1pc, cm

    tr_acc,  tr_f1m,  tr_f1w,  tr_rec,  tr_prec,  tr_f1pc,  tr_cm  = \
        compute_metrics(all_true_train, all_preds_train)
    val_acc, val_f1m, val_f1w, val_rec, val_prec, val_f1pc, val_cm = \
        compute_metrics(all_true_val,   all_preds_val)
    te_acc,  te_f1m,  te_f1w,  te_rec,  te_prec,  te_f1pc,  te_cm  = \
        compute_metrics(all_true_test,  all_preds_test)

    # Per-class shortcuts
    val_rec_sabotage  = val_rec[2]  if len(val_rec)  > 2 else 0.0
    val_rec_exfil     = val_rec[1]  if len(val_rec)  > 1 else 0.0
    val_prec_exfil    = val_prec[1] if len(val_prec) > 1 else 0.0
    val_f1_sabotage   = val_f1pc[2] if len(val_f1pc) > 2 else 0.0

    # ── PRINT ──────────────────────────────────────────────
    print(f"\n  Loss → Train: {train_loss:.4f}  "
          f"Val: {val_loss:.4f}  "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    # Header
    print(f"\n  {'Metric':<24} {'Train':>8} {'Val':>8} {'Test':>8}")
    print(f"  {'-'*50}")

    # Aggregate
    print(f"  {'Accuracy':<24} {tr_acc:>8.4f} {val_acc:>8.4f} {te_acc:>8.4f}")
    print(f"  {'Macro F1':<24} {tr_f1m:>8.4f} {val_f1m:>8.4f} {te_f1m:>8.4f}")
    print(f"  {'Weighted F1':<24} {tr_f1w:>8.4f} {val_f1w:>8.4f} {te_f1w:>8.4f}")

    # Recall
    print(f"  {'-'*50}")
    print(f"  {'Normal recall':<24} "
          f"{tr_rec[0]:>8.4f} {val_rec[0]:>8.4f} {te_rec[0]:>8.4f}")
    print(f"  {'Exfil recall':<24} "
          f"{tr_rec[1] if len(tr_rec)>1 else 0:>8.4f} "
          f"{val_rec_exfil:>8.4f} "
          f"{te_rec[1] if len(te_rec)>1 else 0:>8.4f}")
    print(f"  {'Sabotage recall':<24} "
          f"{tr_rec[2] if len(tr_rec)>2 else 0:>8.4f} "
          f"{val_rec_sabotage:>8.4f} "
          f"{te_rec[2] if len(te_rec)>2 else 0:>8.4f}")

    # Precision
    print(f"  {'-'*50}")
    print(f"  {'Normal precision':<24} "
          f"{tr_prec[0]:>8.4f} {val_prec[0]:>8.4f} {te_prec[0]:>8.4f}")
    print(f"  {'Exfil precision':<24} "
          f"{tr_prec[1] if len(tr_prec)>1 else 0:>8.4f} "
          f"{val_prec_exfil:>8.4f} "
          f"{te_prec[1] if len(te_prec)>1 else 0:>8.4f}")
    print(f"  {'Sabotage precision':<24} "
          f"{tr_prec[2] if len(tr_prec)>2 else 0:>8.4f} "
          f"{val_prec[2] if len(val_prec)>2 else 0:>8.4f} "
          f"{te_prec[2] if len(te_prec)>2 else 0:>8.4f}")

    # Per-class F1
    print(f"  {'-'*50}")
    print(f"  {'Normal F1':<24} "
          f"{tr_f1pc[0]:>8.4f} {val_f1pc[0]:>8.4f} {te_f1pc[0]:>8.4f}")
    print(f"  {'Exfil F1':<24} "
          f"{tr_f1pc[1] if len(tr_f1pc)>1 else 0:>8.4f} "
          f"{val_f1pc[1] if len(val_f1pc)>1 else 0:>8.4f} "
          f"{te_f1pc[1] if len(te_f1pc)>1 else 0:>8.4f}")
    print(f"  {'Sabotage F1':<24} "
          f"{tr_f1pc[2] if len(tr_f1pc)>2 else 0:>8.4f} "
          f"{val_f1_sabotage:>8.4f} "
          f"{te_f1pc[2] if len(te_f1pc)>2 else 0:>8.4f}")

    # Confusion matrices
    print(f"\n  TRAIN CM (rows=true, cols=pred) [Normal|Exfil|Sabotage]:")
    print(f"  {tr_cm}")
    print(f"\n  VAL CM:")
    print(f"  {val_cm}")
    print(f"\n  TEST CM:")
    print(f"  {te_cm}")

    # ── MODEL SELECTION ────────────────────────────────────
    # 1. sabotage recall
    # 2. exfil recall
    # 3. sabotage F1   (balances precision without sacrificing recall)
    # 4. exfil precision
    # 5. val loss

    save_model = False

    if val_rec_sabotage > best_val_sabotage_recall:
       save_model = True

    elif val_rec_sabotage == best_val_sabotage_recall:

        if val_rec_exfil > best_val_exfil_recall:
            save_model = True

        elif val_rec_exfil == best_val_exfil_recall:

            if val_f1_sabotage > best_val_sabotage_f1:
               save_model = True
    if save_model:
        best_val_sabotage_recall = val_rec_sabotage
        best_val_exfil_recall    = val_rec_exfil
        best_val_sabotage_f1     = val_f1_sabotage
        best_val_exfil_precision = val_prec_exfil
        best_val_loss            = val_loss
        patience_counter         = 0
        torch.save(model.state_dict(), "best_model.pt")
        print(f"\n  ✅ Best model saved  "
              f"(sab_rec={val_rec_sabotage:.3f}  "
              f"exfil_rec={val_rec_exfil:.3f}  "
              f"sab_f1={val_f1_sabotage:.3f}  "
              f"exfil_prec={val_prec_exfil:.3f})")
    else:
        patience_counter += 1
        print(f"\n  No improvement — patience {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print("\n  ⛔ Early stopping triggered.")
        break

print(f"\n{'='*52}")
print(f"  Training complete")
print(f"  Best → sab_rec={best_val_sabotage_recall:.3f}  "
      f"exfil_rec={best_val_exfil_recall:.3f}")
print(f"         sab_f1={best_val_sabotage_f1:.3f}  "
      f"exfil_prec={best_val_exfil_precision:.3f}")
print(f"{'='*52}")

print(f"\n{'='*52}")
print(f"  Training complete")
print(f"  Best → sab_rec={best_val_sabotage_recall:.3f}  "
      f"exfil_rec={best_val_exfil_recall:.3f}")
print(f"         sab_prec={best_val_sabotage_precision:.3f}  "
      f"exfil_prec={best_val_exfil_precision:.3f}")
print(f"{'='*52}")

  Val batch 1500/153631
  Loss → Train: 2.3794  Val: 1.2945  LR: 0.000500

  Metric                      Train      Val     Test
  --------------------------------------------------
  Accuracy                   0.7749   0.9333   0.9422
  Macro F1                   0.2961   0.3231   0.3243
  Weighted F1                0.8666   0.9587   0.9684
  --------------------------------------------------
  Normal recall              0.7863   0.9415   0.9448
  Exfil recall               0.0006   0.0000   0.0000
  Sabotage recall            0.9732   0.7500   0.6250
  --------------------------------------------------
  Normal precision           0.9995   0.9945   0.9989
  Exfil precision            0.0023   0.0000   0.0000
  Sabotage precision         0.0036   0.0010   0.0009
  --------------------------------------------------
  Normal F1                  0.8801   0.9673   0.9711
  Exfil F1                   0.0010   0.0000   0.0000
  Sabotage F1                0.0071   0.0020   0.0018

  TRAIN CM

In [99]:
# ============================================================
# THRESHOLD GRID SEARCH — TARGET-BASED SELECTION
# ============================================================
# Step 1 range : 0.10 → 0.15
# Step 2 range : 0.30 → 0.40
#
# Val targets:
#   accuracy        >= 0.90
#   sabotage recall >= 0.75
#   sabotage prec   >= 0.20
#   exfil recall    >= 0.60
#   exfil prec      >= 0.20
#   normal recall   >= 0.85
#
# Test targets:
#   sabotage recall >= 0.50
#   sabotage prec   >= 0.20
#   exfil recall    >= 0.65
#   exfil prec      >= 0.10
#   normal recall   >= 0.85
# ============================================================

import numpy as np
import pandas as pd
import torch
import json
import shutil
import os
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

# ── Load best model ───────────────────────────────────────
threshold_model = TwoStepLSTM(
    input_dim  = X_train.shape[2],
    hidden_dim = 64,
    num_layers = 2,
    dropout    = 0.3
).to(device)
threshold_model.load_state_dict(
    torch.load("best_model.pt", map_location=device)
)
threshold_model.eval()
print("✓ Loaded best_model.pt")


def collect_probs(model, loader):
    s1_list, s2_list, y_list = [], [], []
    with torch.no_grad():
        for X, y1, y2, y_orig in loader:
            s1, s2, _ = model(X)
            s1_list.append(torch.sigmoid(s1).squeeze().cpu())
            s2_list.append(s2.cpu())
            y_list.append(y_orig.cpu())
    return (
        torch.cat(s1_list).numpy(),
        torch.cat(s2_list).numpy(),
        torch.cat(y_list).numpy()
    )


def apply_thresholds(s1_probs, s2_logits, t1, t2):
    final_pred     = np.zeros(len(s1_probs), dtype=int)
    threat_indices = np.where(s1_probs >= t1)[0]
    if len(threat_indices) > 0:
        s2_probs      = torch.softmax(
            torch.tensor(s2_logits[threat_indices]), dim=1
        ).numpy()
        pred_sabotage = (s2_probs[:, 1] >= t2).astype(int)
        final_pred[threat_indices] = np.where(pred_sabotage == 1, 2, 1)
    return final_pred


def get_all_metrics(y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred,
                        labels=[0,1,2], average=None, zero_division=0)
    prec = precision_score(y_true, y_pred,
                           labels=[0,1,2], average=None, zero_division=0)
    f1pc = f1_score(y_true, y_pred,
                    labels=[0,1,2], average=None, zero_division=0)
    f1m  = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    f1w  = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    return acc, rec, prec, f1pc, f1m, f1w, cm


# ── Collect probabilities once ────────────────────────────
print("Collecting val  probabilities...")
s1_val, s2_val, y_val = collect_probs(threshold_model, val_loader)

print("Collecting test probabilities...")
s1_te,  s2_te,  y_te  = collect_probs(threshold_model, test_loader)

# ── Grid search ───────────────────────────────────────────
S1_THRESHOLDS = np.round(np.arange(0.10, 0.16, 0.01), 2)
S2_THRESHOLDS = np.round(np.arange(0.30, 0.41, 0.01), 2)
total_combos  = len(S1_THRESHOLDS) * len(S2_THRESHOLDS)

print(f"\nSearching {len(S1_THRESHOLDS)} × {len(S2_THRESHOLDS)} "
      f"= {total_combos} combinations...")
print(f"\n{'─'*95}")
print(f"  {'t1':>5} {'t2':>5} │ "
      f"{'VAL':^42} │ "
      f"{'TEST':^32} │ "
      f"{'meets':>6}")
print(f"  {'':>5} {'':>5} │ "
      f"{'acc':>6} {'s_rec':>6} {'s_prec':>7} {'e_rec':>6} {'e_prec':>7} {'n_rec':>6} │ "
      f"{'s_rec':>6} {'s_prec':>7} {'e_rec':>6} {'e_prec':>7} {'n_rec':>6} │ "
      f"{'targets':>7}")
print(f"{'─'*95}")

results      = []
best_combo   = None
best_score   = -1

# Val targets
VAL_TARGETS = {
    "acc"    : 0.90,
    "s_rec"  : 0.75,
    "s_prec" : 0.20,
    "e_rec"  : 0.60,
    "e_prec" : 0.20,
    "n_rec"  : 0.85,
}

# Test targets
TEST_TARGETS = {
    "s_rec"  : 0.50,
    "s_prec" : 0.20,
    "e_rec"  : 0.65,
    "e_prec" : 0.10,
    "n_rec"  : 0.85,
}

for t1 in S1_THRESHOLDS:
    for t2 in S2_THRESHOLDS:

        # Val
        pred_val = apply_thresholds(s1_val, s2_val, t1, t2)
        v_acc, v_rec, v_prec, v_f1pc, v_f1m, v_f1w, v_cm = \
            get_all_metrics(y_val, pred_val)

        v_s_rec  = v_rec[2]  if len(v_rec)  > 2 else 0.0
        v_s_prec = v_prec[2] if len(v_prec) > 2 else 0.0
        v_e_rec  = v_rec[1]  if len(v_rec)  > 1 else 0.0
        v_e_prec = v_prec[1] if len(v_prec) > 1 else 0.0
        v_n_rec  = v_rec[0]

        # Test
        pred_te = apply_thresholds(s1_te, s2_te, t1, t2)
        te_acc, te_rec, te_prec, te_f1pc, te_f1m, te_f1w, te_cm = \
            get_all_metrics(y_te, pred_te)

        t_s_rec  = te_rec[2]  if len(te_rec)  > 2 else 0.0
        t_s_prec = te_prec[2] if len(te_prec) > 2 else 0.0
        t_e_rec  = te_rec[1]  if len(te_rec)  > 1 else 0.0
        t_e_prec = te_prec[1] if len(te_prec) > 1 else 0.0
        t_n_rec  = te_rec[0]

        # Check val targets
        val_checks = {
            "acc"    : v_acc    >= VAL_TARGETS["acc"],
            "s_rec"  : v_s_rec  >= VAL_TARGETS["s_rec"],
            "s_prec" : v_s_prec >= VAL_TARGETS["s_prec"],
            "e_rec"  : v_e_rec  >= VAL_TARGETS["e_rec"],
            "e_prec" : v_e_prec >= VAL_TARGETS["e_prec"],
            "n_rec"  : v_n_rec  >= VAL_TARGETS["n_rec"],
        }

        # Check test targets
        test_checks = {
            "s_rec"  : t_s_rec  >= TEST_TARGETS["s_rec"],
            "s_prec" : t_s_prec >= TEST_TARGETS["s_prec"],
            "e_rec"  : t_e_rec  >= TEST_TARGETS["e_rec"],
            "e_prec" : t_e_prec >= TEST_TARGETS["e_prec"],
            "n_rec"  : t_n_rec  >= TEST_TARGETS["n_rec"],
        }

        val_met  = sum(val_checks.values())
        test_met = sum(test_checks.values())
        total_met = val_met + test_met
        all_met   = (val_met == len(val_checks) and
                     test_met == len(test_checks))

        # Score for ranking (weighted sum — prioritise critical metrics)
        score = (
            v_s_rec  * 3.0 +   # sabotage recall most important
            t_s_rec  * 3.0 +
            v_e_rec  * 2.0 +
            t_e_rec  * 2.0 +
            v_s_prec * 1.5 +
            t_s_prec * 1.5 +
            v_acc    * 1.0 +
            v_e_prec * 1.0 +
            t_e_prec * 1.0 +
            v_n_rec  * 0.5 +
            t_n_rec  * 0.5
        )

        meets_str = f"{total_met}/{len(val_checks)+len(test_checks)}"
        if all_met:
            meets_str = "ALL ✓"

        print(f"  {t1:>5.2f} {t2:>5.2f} │ "
              f"{v_acc:>6.3f} {v_s_rec:>6.3f} {v_s_prec:>7.3f} "
              f"{v_e_rec:>6.3f} {v_e_prec:>7.3f} {v_n_rec:>6.3f} │ "
              f"{t_s_rec:>6.3f} {t_s_prec:>7.3f} "
              f"{t_e_rec:>6.3f} {t_e_prec:>7.3f} {t_n_rec:>6.3f} │ "
              f"{meets_str:>7}")

        results.append({
            "t1"         : t1,   "t2"         : t2,
            "val_acc"    : round(v_acc,    4),
            "val_s_rec"  : round(v_s_rec,  4),
            "val_s_prec" : round(v_s_prec, 4),
            "val_e_rec"  : round(v_e_rec,  4),
            "val_e_prec" : round(v_e_prec, 4),
            "val_n_rec"  : round(v_n_rec,  4),
            "val_f1_mac" : round(v_f1m,    4),
            "test_s_rec" : round(t_s_rec,  4),
            "test_s_prec": round(t_s_prec, 4),
            "test_e_rec" : round(t_e_rec,  4),
            "test_e_prec": round(t_e_prec, 4),
            "test_n_rec" : round(t_n_rec,  4),
            "test_f1_mac": round(te_f1m,   4),
            "val_met"    : val_met,
            "test_met"   : test_met,
            "total_met"  : total_met,
            "all_met"    : all_met,
            "score"      : round(score, 4),
            # confusion matrices
            "v_cm_00": int(v_cm[0,0]), "v_cm_01": int(v_cm[0,1]), "v_cm_02": int(v_cm[0,2]),
            "v_cm_10": int(v_cm[1,0]), "v_cm_11": int(v_cm[1,1]), "v_cm_12": int(v_cm[1,2]),
            "v_cm_20": int(v_cm[2,0]), "v_cm_21": int(v_cm[2,1]), "v_cm_22": int(v_cm[2,2]),
            "t_cm_00": int(te_cm[0,0]),"t_cm_01": int(te_cm[0,1]),"t_cm_02": int(te_cm[0,2]),
            "t_cm_10": int(te_cm[1,0]),"t_cm_11": int(te_cm[1,1]),"t_cm_12": int(te_cm[1,2]),
            "t_cm_20": int(te_cm[2,0]),"t_cm_21": int(te_cm[2,1]),"t_cm_22": int(te_cm[2,2]),
        })

        if score > best_score:
            best_score = score
            best_combo = results[-1]

print(f"{'─'*95}")

# ── Sort results ──────────────────────────────────────────
results_df = pd.DataFrame(results).sort_values(
    by=["all_met", "total_met", "score"],
    ascending=False
).reset_index(drop=True)

# ── Pick best ─────────────────────────────────────────────
# Prefer a combo that meets all targets
# If none meet all → pick highest score
all_met_df = results_df[results_df["all_met"] == True]
if len(all_met_df) > 0:
    best = all_met_df.iloc[0]
    print(f"\n✓ Found {len(all_met_df)} combos meeting ALL targets")
else:
    best = results_df.iloc[0]
    print(f"\n⚠ No combo meets ALL targets — picking best score")
    print(f"  Best meets {best['total_met']}/{len(val_checks)+len(test_checks)} targets")

BEST_T1 = best["t1"]
BEST_T2 = best["t2"]

# ── Print best result ─────────────────────────────────────
print(f"\n{'='*60}")
print(f"  BEST COMBINATION  t1={BEST_T1}  t2={BEST_T2}")
print(f"{'='*60}")
print(f"\n  {'Metric':<24} {'Val':>8} {'Target':>8} {'Met':>5} │ "
      f"{'Test':>8} {'Target':>8} {'Met':>5}")
print(f"  {'-'*70}")

checks = [
    ("Accuracy",        best["val_acc"],    VAL_TARGETS["acc"],    None,                None),
    ("Sabotage recall", best["val_s_rec"],  VAL_TARGETS["s_rec"],  best["test_s_rec"],  TEST_TARGETS["s_rec"]),
    ("Sabotage prec",   best["val_s_prec"], VAL_TARGETS["s_prec"], best["test_s_prec"], TEST_TARGETS["s_prec"]),
    ("Exfil recall",    best["val_e_rec"],  VAL_TARGETS["e_rec"],  best["test_e_rec"],  TEST_TARGETS["e_rec"]),
    ("Exfil prec",      best["val_e_prec"], VAL_TARGETS["e_prec"], best["test_e_prec"], TEST_TARGETS["e_prec"]),
    ("Normal recall",   best["val_n_rec"],  VAL_TARGETS["n_rec"],  best["test_n_rec"],  TEST_TARGETS["n_rec"]),
]

for name, v_val, v_tgt, t_val, t_tgt in checks:
    v_met = "✓" if v_val >= v_tgt else "✗"
    if t_val is not None:
        t_met = "✓" if t_val >= t_tgt else "✗"
        print(f"  {name:<24} {v_val:>8.4f} {v_tgt:>8.2f} {v_met:>5} │ "
              f"{t_val:>8.4f} {t_tgt:>8.2f} {t_met:>5}")
    else:
        print(f"  {name:<24} {v_val:>8.4f} {v_tgt:>8.2f} {v_met:>5} │ "
              f"{'—':>8} {'—':>8} {'—':>5}")

print(f"\n  Val  CM [Normal|Exfil|Sabotage]:")
print(f"  [[{best['v_cm_00']:>6} {best['v_cm_01']:>6} {best['v_cm_02']:>6}]")
print(f"   [{best['v_cm_10']:>6} {best['v_cm_11']:>6} {best['v_cm_12']:>6}]")
print(f"   [{best['v_cm_20']:>6} {best['v_cm_21']:>6} {best['v_cm_22']:>6}]]")
print(f"\n  Test CM [Normal|Exfil|Sabotage]:")
print(f"  [[{best['t_cm_00']:>6} {best['t_cm_01']:>6} {best['t_cm_02']:>6}]")
print(f"   [{best['t_cm_10']:>6} {best['t_cm_11']:>6} {best['t_cm_12']:>6}]")
print(f"   [{best['t_cm_20']:>6} {best['t_cm_21']:>6} {best['t_cm_22']:>6}]]")

# ── Save ──────────────────────────────────────────────────
results_df.to_csv("threshold_tuning_results.csv", index=False)

threshold_config = {
    "t1_step1"       : float(BEST_T1),
    "t2_step2"       : float(BEST_T2),
    "val_acc"        : float(best["val_acc"]),
    "val_s_rec"      : float(best["val_s_rec"]),
    "val_s_prec"     : float(best["val_s_prec"]),
    "val_e_rec"      : float(best["val_e_rec"]),
    "val_e_prec"     : float(best["val_e_prec"]),
    "val_n_rec"      : float(best["val_n_rec"]),
    "test_s_rec"     : float(best["test_s_rec"]),
    "test_s_prec"    : float(best["test_s_prec"]),
    "test_e_rec"     : float(best["test_e_rec"]),
    "test_e_prec"    : float(best["test_e_prec"]),
    "test_n_rec"     : float(best["test_n_rec"]),
    "targets_met"    : f"{best['total_met']}/{len(val_checks)+len(test_checks)}",
    "all_targets_met": bool(best["all_met"]),
}
with open("threshold_config.json", "w") as f:
    json.dump(threshold_config, f, indent=2)

shutil.copy("best_model.pt", "threshold_best_model.pt")

print(f"\n{'='*60}")
print("FILES SAVED")
print(f"{'='*60}")
print(f"  best_model.pt              — original (unchanged)")
print(f"  threshold_best_model.pt    — copy with best thresholds")
print(f"  threshold_config.json      — thresholds + all metrics")
print(f"  threshold_tuning_results.csv — all {len(results_df)} combinations")

C:\Users\student\AppData\Local\Temp\ipykernel_9728\3242703808.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("best_model.pt", map_location=device)


✓ Loaded best_model.pt

Searching 6 × 11 = 66 combinations...

───────────────────────────────────────────────────────────────────────────────────────────────
     t1    t2 │                    VAL                     │               TEST               │  meets
              │    acc  s_rec  s_prec  e_rec  e_prec  n_rec │  s_rec  s_prec  e_rec  e_prec  n_rec │ targets
───────────────────────────────────────────────────────────────────────────────────────────────
   0.10  0.30 │  0.806  0.750   0.000  0.010   0.004  0.813 │  0.875   0.000  0.011   0.002  0.830 │    2/11
   0.10  0.31 │  0.807  0.750   0.001  0.094   0.007  0.813 │  0.875   0.001  0.069   0.002  0.830 │    2/11
   0.10  0.32 │  0.810  0.750   0.037  0.508   0.024  0.813 │  0.750   0.054  0.673   0.011  0.830 │    3/11
   0.10  0.33 │  0.810  0.750   0.078  0.512   0.024  0.813 │  0.750   0.103  0.676   0.011  0.830 │    3/11
   0.10  0.34 │  0.811  0.750   0.113  0.513   0.024  0.813 │  0.750   0.150  0.676   0.011  0.83

In [101]:
# ============================================================
# CONFUSION MATRIX + METRICS VISUALIZATION
# model: threshold_best_model.pt  (t1=0.10, t2=0.35)
# ============================================================

import numpy as np
import torch
import json
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    classification_report
)

# ── Load thresholds ───────────────────────────────────────
with open("threshold_config.json") as f:
    tcfg = json.load(f)
T1 = tcfg["t1_step1"]
T2 = tcfg["t2_step2"]
print(f"✓ Thresholds → t1={T1}  t2={T2}")

# ── Load model ────────────────────────────────────────────
eval_model = TwoStepLSTM(
    input_dim  = X_train.shape[2],
    hidden_dim = 64,
    num_layers = 2,
    dropout    = 0.3
).to(device)
eval_model.load_state_dict(
    torch.load("threshold_best_model.pt", map_location=device)
)
eval_model.eval()
print("✓ Loaded threshold_best_model.pt")


# ── Inference helper ──────────────────────────────────────
def run_eval(model, loader, t1, t2):
    preds, trues = [], []
    with torch.no_grad():
        for X, y1, y2, y_orig in loader:
            s1, s2, _ = model(X)
            s1p = torch.sigmoid(s1).squeeze().cpu().numpy()
            s2p = torch.softmax(s2, dim=1).cpu().numpy()
            if s1p.ndim == 0:
                s1p = s1p.reshape(1)
            if s2p.ndim == 1:
                s2p = s2p.reshape(1, -1)
            pred = np.zeros(len(s1p), dtype=int)
            mask = s1p >= t1
            if mask.sum() > 0:
                pred[mask] = np.where(s2p[mask, 1] >= t2, 2, 1)
            preds.extend(pred.tolist())
            trues.extend(y_orig.cpu().numpy().tolist())
    return np.array(trues), np.array(preds)


# ── Run on all splits ─────────────────────────────────────
print("\nRunning inference...")
y_tr, p_tr = run_eval(eval_model, train_loader, T1, T2)
y_va, p_va = run_eval(eval_model, val_loader,   T1, T2)
y_te, p_te = run_eval(eval_model, test_loader,  T1, T2)
print("✓ Inference complete")


# ── Metrics helper ────────────────────────────────────────
def full_metrics(y_true, y_pred, split):
    labels     = [0, 1, 2]
    names      = ["Normal", "Exfil", "Sabotage"]
    acc        = accuracy_score(y_true, y_pred)
    rec        = recall_score(y_true, y_pred,    labels=labels, average=None, zero_division=0)
    prec       = precision_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    f1pc       = f1_score(y_true, y_pred,        labels=labels, average=None, zero_division=0)
    f1_mac     = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    f1_wei     = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    cm         = confusion_matrix(y_true, y_pred, labels=labels)

    print(f"\n{'='*60}")
    print(f"  {split}  (t1={T1}  t2={T2})")
    print(f"{'='*60}")
    print(f"  Accuracy   : {acc:.4f}")
    print(f"  Macro F1   : {f1_mac:.4f}")
    print(f"  Weighted F1: {f1_wei:.4f}")
    print(f"\n  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print(f"  {'-'*44}")
    for i, name in enumerate(names):
        sup = (y_true == i).sum()
        print(f"  {name:<12} {prec[i]:>10.4f} {rec[i]:>10.4f} "
              f"{f1pc[i]:>10.4f} {sup:>10,}")
    print(f"\n  Confusion Matrix [rows=true, cols=pred]")
    print(f"  Labels → [Normal, Exfil, Sabotage]")
    print(f"  {cm}")
    return acc, rec, prec, f1pc, f1_mac, f1_wei, cm


# ── Print metrics ─────────────────────────────────────────
tr_acc, tr_rec, tr_prec, tr_f1pc, tr_f1m, tr_f1w, tr_cm = \
    full_metrics(y_tr, p_tr, "TRAIN")
va_acc, va_rec, va_prec, va_f1pc, va_f1m, va_f1w, va_cm = \
    full_metrics(y_va, p_va, "VAL")
te_acc, te_rec, te_prec, te_f1pc, te_f1m, te_f1w, te_cm = \
    full_metrics(y_te, p_te, "TEST")

✓ Thresholds → t1=0.13  t2=0.34
✓ Loaded threshold_best_model.pt

Running inference...


C:\Users\student\AppData\Local\Temp\ipykernel_9728\1867293269.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("threshold_best_model.pt", map_location=device)

✓ Inference complete

  TRAIN  (t1=0.13  t2=0.34)
  Accuracy   : 0.8533
  Macro F1   : 0.4469
  Weighted F1: 0.9080

  Class         Precision     Recall         F1    Support
  --------------------------------------------
  Normal           0.9997     0.8513     0.9196    543,807
  Exfil            0.0919     0.9818     0.1681      8,076
  Sabotage         0.1452     0.9777     0.2529        448

  Confusion Matrix [rows=true, cols=pred]
  Labels → [Normal, Exfil, Sabotage]
  [[462944  78302   2561]
 [   130   7929     17]
 [     2      8    438]]

  VAL  (t1=0.13  t2=0.34)
  Accuracy   : 0.8555
  Macro F1   : 0.4396
  Weighted F1: 0.9141

  Class         Precision     Recall         F1    Support
  --------------------------------------------
  Normal           0.9946     0.8589     0.9218     97,398
  Exfil            0.0288     0.4733     0.0543        860
  Sabotage         0.2222     0.7500     0.3429          8

  Confusion Matrix [rows=true, cols=pred]
  Labels → [Normal, Exfil

In [102]:
# ── Visualization ─────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

CLASS_NAMES  = ["Normal", "Exfil", "Sabotage"]
SPLITS       = ["Train", "Val", "Test"]
CMS          = [tr_cm,   va_cm,  te_cm]
ACCS         = [tr_acc,  va_acc, te_acc]
F1_MACS      = [tr_f1m,  va_f1m, te_f1m]
F1_WEI       = [tr_f1w,  va_f1w, te_f1w]
ALL_REC      = [tr_rec,  va_rec, te_rec]
ALL_PREC     = [tr_prec, va_prec,te_prec]
ALL_F1PC     = [tr_f1pc, va_f1pc,te_f1pc]

# ── Figure layout ─────────────────────────────────────────
# Row 1: confusion matrices (3 cols)
# Row 2: per-class recall bar chart
# Row 3: per-class precision bar chart
# Row 4: per-class F1 + aggregate metrics

fig = plt.figure(figsize=(20, 26))
fig.suptitle(
    f"threshold_best_model.pt  |  t1={T1}  t2={T2}",
    fontsize=14, fontweight='bold', y=0.99
)

gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

colors = {
    "Normal"  : "#4C72B0",
    "Exfil"   : "#DD8452",
    "Sabotage": "#C44E52",
}
split_colors = ["#4C72B0", "#55A868", "#C44E52"]

# ── Row 1: Confusion matrices ─────────────────────────────
for col, (split, cm, acc) in enumerate(zip(SPLITS, CMS, ACCS)):
    ax = fig.add_subplot(gs[0, col])

    # Normalised for colour, raw counts as annotations
    cm_norm = cm.astype(float)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_pct = cm_norm / row_sums

    annot = np.empty_like(cm, dtype=object)
    for i in range(3):
        for j in range(3):
            annot[i, j] = f"{cm[i,j]:,}\n({cm_pct[i,j]*100:.1f}%)"

    sns.heatmap(
        cm_pct, annot=annot, fmt='', cmap='Blues',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        ax=ax, linewidths=0.5, vmin=0, vmax=1,
        cbar_kws={"shrink": 0.8}
    )
    ax.set_title(f"{split}  (acc={acc:.3f})", fontsize=11, fontweight='bold')
    ax.set_xlabel("Predicted", fontsize=9)
    ax.set_ylabel("True",      fontsize=9)
    ax.tick_params(labelsize=8)

# ── Row 2: Per-class recall ───────────────────────────────
ax_rec = fig.add_subplot(gs[1, :])
x      = np.arange(len(CLASS_NAMES))
width  = 0.25

for i, (split, rec) in enumerate(zip(SPLITS, ALL_REC)):
    bars = ax_rec.bar(
        x + i * width, rec, width,
        label=split, color=split_colors[i], alpha=0.85
    )
    for bar, val in zip(bars, rec):
        ax_rec.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f"{val:.3f}", ha='center', va='bottom', fontsize=8
        )

ax_rec.set_title("Per-class Recall", fontsize=11, fontweight='bold')
ax_rec.set_xticks(x + width)
ax_rec.set_xticklabels(CLASS_NAMES, fontsize=10)
ax_rec.set_ylim(0, 1.15)
ax_rec.set_ylabel("Recall")
ax_rec.legend(fontsize=9)
ax_rec.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax_rec.grid(axis='y', alpha=0.3)

# ── Row 3: Per-class precision ────────────────────────────
ax_prec = fig.add_subplot(gs[2, :])

for i, (split, prec) in enumerate(zip(SPLITS, ALL_PREC)):
    bars = ax_prec.bar(
        x + i * width, prec, width,
        label=split, color=split_colors[i], alpha=0.85
    )
    for bar, val in zip(bars, prec):
        ax_prec.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f"{val:.3f}", ha='center', va='bottom', fontsize=8
        )

ax_prec.set_title("Per-class Precision", fontsize=11, fontweight='bold')
ax_prec.set_xticks(x + width)
ax_prec.set_xticklabels(CLASS_NAMES, fontsize=10)
ax_prec.set_ylim(0, 1.15)
ax_prec.set_ylabel("Precision")
ax_prec.legend(fontsize=9)
ax_prec.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax_prec.grid(axis='y', alpha=0.3)

# ── Row 4 left: Per-class F1 ──────────────────────────────
ax_f1pc = fig.add_subplot(gs[3, :2])

for i, (split, f1pc) in enumerate(zip(SPLITS, ALL_F1PC)):
    bars = ax_f1pc.bar(
        x + i * width, f1pc, width,
        label=split, color=split_colors[i], alpha=0.85
    )
    for bar, val in zip(bars, f1pc):
        ax_f1pc.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f"{val:.3f}", ha='center', va='bottom', fontsize=8
        )

ax_f1pc.set_title("Per-class F1", fontsize=11, fontweight='bold')
ax_f1pc.set_xticks(x + width)
ax_f1pc.set_xticklabels(CLASS_NAMES, fontsize=10)
ax_f1pc.set_ylim(0, 1.15)
ax_f1pc.set_ylabel("F1 Score")
ax_f1pc.legend(fontsize=9)
ax_f1pc.grid(axis='y', alpha=0.3)

# ── Row 4 right: Aggregate metrics ───────────────────────
ax_agg = fig.add_subplot(gs[3, 2])

metrics_names = ["Accuracy", "Macro F1", "Weighted F1"]
tr_agg  = [tr_acc, tr_f1m, tr_f1w]
va_agg  = [va_acc, va_f1m, va_f1w]
te_agg  = [te_acc, te_f1m, te_f1w]

xm     = np.arange(len(metrics_names))
wm     = 0.25

for i, (split, vals) in enumerate(zip(SPLITS, [tr_agg, va_agg, te_agg])):
    bars = ax_agg.bar(
        xm + i * wm, vals, wm,
        label=split, color=split_colors[i], alpha=0.85
    )
    for bar, val in zip(bars, vals):
        ax_agg.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f"{val:.3f}", ha='center', va='bottom', fontsize=7
        )

ax_agg.set_title("Aggregate Metrics", fontsize=11, fontweight='bold')
ax_agg.set_xticks(xm + wm)
ax_agg.set_xticklabels(metrics_names, fontsize=8)
ax_agg.set_ylim(0, 1.15)
ax_agg.set_ylabel("Score")
ax_agg.legend(fontsize=8)
ax_agg.grid(axis='y', alpha=0.3)

plt.savefig("threshold_best_model_metrics.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: threshold_best_model_metrics.png")


Saved: threshold_best_model_metrics.png


C:\Users\student\AppData\Local\Temp\ipykernel_9728\4218884611.py:171: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [100]:
# ============================================================
# THRESHOLD GRID SEARCH — 7 TARGET CONDITIONS
# ============================================================
# Step 1 range : 0.10 → 0.15
# Step 2 range : 0.30 → 0.40
#
# Val targets  (4):
#   accuracy        >= 0.90
#   sabotage recall >= 0.75
#   exfil recall    >= 0.60
#   normal recall   >= 0.85
#
# Test targets (3):
#   sabotage recall >= 0.50
#   exfil recall    >= 0.65
#   normal recall   >= 0.85
#
# Total = 7 conditions
# ============================================================

import numpy as np
import pandas as pd
import torch
import json
import shutil
import os
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

# ── Load best model ───────────────────────────────────────
threshold_model = TwoStepLSTM(
    input_dim  = X_train.shape[2],
    hidden_dim = 64,
    num_layers = 2,
    dropout    = 0.3
).to(device)
threshold_model.load_state_dict(
    torch.load("best_model.pt", map_location=device)
)
threshold_model.eval()
print("✓ Loaded best_model.pt")


def collect_probs(model, loader):
    s1_list, s2_list, y_list = [], [], []
    with torch.no_grad():
        for X, y1, y2, y_orig in loader:
            s1, s2, _ = model(X)
            s1_list.append(torch.sigmoid(s1).squeeze().cpu())
            s2_list.append(s2.cpu())
            y_list.append(y_orig.cpu())
    return (
        torch.cat(s1_list).numpy(),
        torch.cat(s2_list).numpy(),
        torch.cat(y_list).numpy()
    )


def apply_thresholds(s1_probs, s2_logits, t1, t2):
    final_pred     = np.zeros(len(s1_probs), dtype=int)
    threat_indices = np.where(s1_probs >= t1)[0]
    if len(threat_indices) > 0:
        s2_probs      = torch.softmax(
            torch.tensor(s2_logits[threat_indices]), dim=1
        ).numpy()
        pred_sabotage = (s2_probs[:, 1] >= t2).astype(int)
        final_pred[threat_indices] = np.where(pred_sabotage == 1, 2, 1)
    return final_pred


def get_all_metrics(y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred,
                        labels=[0,1,2], average=None, zero_division=0)
    prec = precision_score(y_true, y_pred,
                           labels=[0,1,2], average=None, zero_division=0)
    f1pc = f1_score(y_true, y_pred,
                    labels=[0,1,2], average=None, zero_division=0)
    f1m  = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    f1w  = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    return acc, rec, prec, f1pc, f1m, f1w, cm


# ── Collect probabilities once ────────────────────────────
print("Collecting val  probabilities...")
s1_val, s2_val, y_val = collect_probs(threshold_model, val_loader)

print("Collecting test probabilities...")
s1_te,  s2_te,  y_te  = collect_probs(threshold_model, test_loader)

# ── 7 Targets ─────────────────────────────────────────────
VAL_TARGETS = {
    "acc"   : 0.90,
    "s_rec" : 0.75,
    "e_rec" : 0.60,
    "n_rec" : 0.85,
}
TEST_TARGETS = {
    "s_rec" : 0.50,
    "e_rec" : 0.65,
    "n_rec" : 0.85,
}
TOTAL_CONDITIONS = len(VAL_TARGETS) + len(TEST_TARGETS)  # 7

# ── Grid search ───────────────────────────────────────────
S1_THRESHOLDS = np.round(np.arange(0.10, 0.16, 0.01), 2)
S2_THRESHOLDS = np.round(np.arange(0.30, 0.41, 0.01), 2)
total_combos  = len(S1_THRESHOLDS) * len(S2_THRESHOLDS)

print(f"\nSearching {len(S1_THRESHOLDS)} × {len(S2_THRESHOLDS)} "
      f"= {total_combos} combinations  ({TOTAL_CONDITIONS} conditions)\n")

print(f"{'─'*90}")
print(f"  {'t1':>5} {'t2':>5} │ "
      f"{'VAL':^36} │ "
      f"{'TEST':^26} │ "
      f"{'met/7':>6}")
print(f"  {'':>5} {'':>5} │ "
      f"{'acc':>6} {'s_rec':>6} {'e_rec':>6} {'n_rec':>6} "
      f"{'s_prec':>7} {'e_prec':>7} │ "
      f"{'s_rec':>6} {'e_rec':>6} {'n_rec':>6} "
      f"{'s_prec':>7} │ "
      f"{'':>6}")
print(f"{'─'*90}")

results    = []
best_score = -1

for t1 in S1_THRESHOLDS:
    for t2 in S2_THRESHOLDS:

        # Val metrics
        pred_val = apply_thresholds(s1_val, s2_val, t1, t2)
        v_acc, v_rec, v_prec, v_f1pc, v_f1m, v_f1w, v_cm = \
            get_all_metrics(y_val, pred_val)

        v_s_rec  = v_rec[2]  if len(v_rec)  > 2 else 0.0
        v_s_prec = v_prec[2] if len(v_prec) > 2 else 0.0
        v_e_rec  = v_rec[1]  if len(v_rec)  > 1 else 0.0
        v_e_prec = v_prec[1] if len(v_prec) > 1 else 0.0
        v_n_rec  = v_rec[0]

        # Test metrics
        pred_te = apply_thresholds(s1_te, s2_te, t1, t2)
        te_acc, te_rec, te_prec, te_f1pc, te_f1m, te_f1w, te_cm = \
            get_all_metrics(y_te, pred_te)

        t_s_rec  = te_rec[2]  if len(te_rec)  > 2 else 0.0
        t_s_prec = te_prec[2] if len(te_prec) > 2 else 0.0
        t_e_rec  = te_rec[1]  if len(te_rec)  > 1 else 0.0
        t_e_prec = te_prec[1] if len(te_prec) > 1 else 0.0
        t_n_rec  = te_rec[0]

        # ── Check 7 conditions ────────────────────────────
        val_checks = {
            "acc"   : v_acc   >= VAL_TARGETS["acc"],
            "s_rec" : v_s_rec >= VAL_TARGETS["s_rec"],
            "e_rec" : v_e_rec >= VAL_TARGETS["e_rec"],
            "n_rec" : v_n_rec >= VAL_TARGETS["n_rec"],
        }
        test_checks = {
            "s_rec" : t_s_rec >= TEST_TARGETS["s_rec"],
            "e_rec" : t_e_rec >= TEST_TARGETS["e_rec"],
            "n_rec" : t_n_rec >= TEST_TARGETS["n_rec"],
        }

        val_met   = sum(val_checks.values())
        test_met  = sum(test_checks.values())
        total_met = val_met + test_met
        all_met   = (total_met == TOTAL_CONDITIONS)

        # Score — precision shown but not in conditions
        score = (
            v_s_rec  * 3.0 +
            t_s_rec  * 3.0 +
            v_e_rec  * 2.0 +
            t_e_rec  * 2.0 +
            v_acc    * 1.0 +
            v_n_rec  * 0.5 +
            t_n_rec  * 0.5 +
            v_s_prec * 0.3 +   # tracked but not in conditions
            t_s_prec * 0.3 +
            v_e_prec * 0.2 +
            t_e_prec * 0.2
        )

        meets_str = f"{total_met}/7"
        if all_met:
            meets_str = "7/7 ✓"

        print(f"  {t1:>5.2f} {t2:>5.2f} │ "
              f"{v_acc:>6.3f} {v_s_rec:>6.3f} {v_e_rec:>6.3f} {v_n_rec:>6.3f} "
              f"{v_s_prec:>7.3f} {v_e_prec:>7.3f} │ "
              f"{t_s_rec:>6.3f} {t_e_rec:>6.3f} {t_n_rec:>6.3f} "
              f"{t_s_prec:>7.3f} │ "
              f"{meets_str:>6}")

        results.append({
            "t1"         : t1,
            "t2"         : t2,
            "val_acc"    : round(v_acc,    4),
            "val_s_rec"  : round(v_s_rec,  4),
            "val_e_rec"  : round(v_e_rec,  4),
            "val_n_rec"  : round(v_n_rec,  4),
            "val_s_prec" : round(v_s_prec, 4),
            "val_e_prec" : round(v_e_prec, 4),
            "val_f1_mac" : round(v_f1m,    4),
            "test_s_rec" : round(t_s_rec,  4),
            "test_e_rec" : round(t_e_rec,  4),
            "test_n_rec" : round(t_n_rec,  4),
            "test_s_prec": round(t_s_prec, 4),
            "test_e_prec": round(t_e_prec, 4),
            "test_f1_mac": round(te_f1m,   4),
            "val_met"    : val_met,
            "test_met"   : test_met,
            "total_met"  : total_met,
            "all_met"    : all_met,
            "score"      : round(score,    4),
            "v_cm_00": int(v_cm[0,0]),  "v_cm_01": int(v_cm[0,1]),  "v_cm_02": int(v_cm[0,2]),
            "v_cm_10": int(v_cm[1,0]),  "v_cm_11": int(v_cm[1,1]),  "v_cm_12": int(v_cm[1,2]),
            "v_cm_20": int(v_cm[2,0]),  "v_cm_21": int(v_cm[2,1]),  "v_cm_22": int(v_cm[2,2]),
            "t_cm_00": int(te_cm[0,0]), "t_cm_01": int(te_cm[0,1]), "t_cm_02": int(te_cm[0,2]),
            "t_cm_10": int(te_cm[1,0]), "t_cm_11": int(te_cm[1,1]), "t_cm_12": int(te_cm[1,2]),
            "t_cm_20": int(te_cm[2,0]), "t_cm_21": int(te_cm[2,1]), "t_cm_22": int(te_cm[2,2]),
        })

        if score > best_score:
            best_score = score

print(f"{'─'*90}")

# ── Sort + pick best ──────────────────────────────────────
results_df = pd.DataFrame(results).sort_values(
    by=["all_met", "total_met", "score"],
    ascending=False
).reset_index(drop=True)

all_met_df = results_df[results_df["all_met"] == True]
if len(all_met_df) > 0:
    best = all_met_df.iloc[0]
    print(f"\n✓ Found {len(all_met_df)} combo(s) meeting all 7 conditions")
else:
    best = results_df.iloc[0]
    print(f"\n⚠ No combo meets all 7 — picking best score")
    print(f"  Best meets {best['total_met']}/7 conditions")

BEST_T1 = best["t1"]
BEST_T2 = best["t2"]

# ── Print best ────────────────────────────────────────────
print(f"\n{'='*62}")
print(f"  BEST  t1={BEST_T1}  t2={BEST_T2}  "
      f"({best['total_met']}/7 conditions met)")
print(f"{'='*62}")
print(f"\n  {'Condition':<24} {'Val':>8} {'Target':>8} {'✓?':>4} │ "
      f"{'Test':>8} {'Target':>8} {'✓?':>4}")
print(f"  {'-'*68}")

condition_rows = [
    ("Accuracy (val)",    best["val_acc"],   VAL_TARGETS["acc"],   None,               None),
    ("Sabotage recall",   best["val_s_rec"], VAL_TARGETS["s_rec"], best["test_s_rec"], TEST_TARGETS["s_rec"]),
    ("Exfil recall",      best["val_e_rec"], VAL_TARGETS["e_rec"], best["test_e_rec"], TEST_TARGETS["e_rec"]),
    ("Normal recall",     best["val_n_rec"], VAL_TARGETS["n_rec"], best["test_n_rec"], TEST_TARGETS["n_rec"]),
]
info_rows = [
    ("Sabotage prec",     best["val_s_prec"], "—", best["test_s_prec"], "—"),
    ("Exfil prec",        best["val_e_prec"], "—", best["test_e_prec"], "—"),
    ("Macro F1",          best["val_f1_mac"], "—", best["test_f1_mac"], "—"),
]

for name, v_val, v_tgt, t_val, t_tgt in condition_rows:
    v_met = "✓" if v_val >= v_tgt else "✗"
    if t_val is not None:
        t_met = "✓" if t_val >= t_tgt else "✗"
        print(f"  {name:<24} {v_val:>8.4f} {v_tgt:>8.2f} {v_met:>4} │ "
              f"{t_val:>8.4f} {t_tgt:>8.2f} {t_met:>4}")
    else:
        print(f"  {name:<24} {v_val:>8.4f} {v_tgt:>8.2f} {v_met:>4} │ "
              f"{'—':>8} {'—':>8} {'—':>4}")

print(f"  {'─'*68}")
print(f"  (info only — not in conditions)")
for name, v_val, v_tgt, t_val, t_tgt in info_rows:
    print(f"  {name:<24} {v_val:>8.4f} {str(v_tgt):>8} {'':>4} │ "
          f"{t_val:>8.4f} {str(t_tgt):>8} {'':>4}")

print(f"\n  Val  CM [Normal|Exfil|Sabotage]:")
print(f"  [[{best['v_cm_00']:>6} {best['v_cm_01']:>6} {best['v_cm_02']:>6}]")
print(f"   [{best['v_cm_10']:>6} {best['v_cm_11']:>6} {best['v_cm_12']:>6}]")
print(f"   [{best['v_cm_20']:>6} {best['v_cm_21']:>6} {best['v_cm_22']:>6}]]")
print(f"\n  Test CM [Normal|Exfil|Sabotage]:")
print(f"  [[{best['t_cm_00']:>6} {best['t_cm_01']:>6} {best['t_cm_02']:>6}]")
print(f"   [{best['t_cm_10']:>6} {best['t_cm_11']:>6} {best['t_cm_12']:>6}]")
print(f"   [{best['t_cm_20']:>6} {best['t_cm_21']:>6} {best['t_cm_22']:>6}]]")

# ── Save ──────────────────────────────────────────────────
results_df.to_csv("threshold_tuning_7cond_results.csv", index=False)

threshold_config = {
    "t1_step1"         : float(BEST_T1),
    "t2_step2"         : float(BEST_T2),
    "conditions"       : "7",
    "val_acc"          : float(best["val_acc"]),
    "val_s_rec"        : float(best["val_s_rec"]),
    "val_e_rec"        : float(best["val_e_rec"]),
    "val_n_rec"        : float(best["val_n_rec"]),
    "val_s_prec"       : float(best["val_s_prec"]),
    "val_e_prec"       : float(best["val_e_prec"]),
    "val_f1_macro"     : float(best["val_f1_mac"]),
    "test_s_rec"       : float(best["test_s_rec"]),
    "test_e_rec"       : float(best["test_e_rec"]),
    "test_n_rec"       : float(best["test_n_rec"]),
    "test_s_prec"      : float(best["test_s_prec"]),
    "test_e_prec"      : float(best["test_e_prec"]),
    "test_f1_macro"    : float(best["test_f1_mac"]),
    "conditions_met"   : f"{best['total_met']}/7",
    "all_conditions_met": bool(best["all_met"]),
}
with open("threshold_7cond_config.json", "w") as f:
    json.dump(threshold_config, f, indent=2)

# ── Save model copy ───────────────────────────────────────
shutil.copy("best_model.pt", "threshold2_with7conditions.pt")

print(f"\n{'='*62}")
print("FILES SAVED")
print(f"{'='*62}")
print(f"  best_model.pt                    — original (unchanged)")
print(f"  threshold2_with7conditions.pt    — model with 7-condition thresholds")
print(f"  threshold_7cond_config.json      — t1={BEST_T1} t2={BEST_T2} + all metrics")
print(f"  threshold_tuning_7cond_results.csv — all {len(results_df)} combinations")

C:\Users\student\AppData\Local\Temp\ipykernel_9728\3745571285.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("best_model.pt", map_location=device)


✓ Loaded best_model.pt

Searching 6 × 11 = 66 combinations  (7 conditions)

──────────────────────────────────────────────────────────────────────────────────────────
     t1    t2 │                 VAL                  │            TEST            │  met/7
              │    acc  s_rec  e_rec  n_rec  s_prec  e_prec │  s_rec  e_rec  n_rec  s_prec │       
──────────────────────────────────────────────────────────────────────────────────────────
   0.10  0.30 │  0.806  0.750  0.010  0.813   0.000   0.004 │  0.875  0.011  0.830   0.000 │    2/7
   0.10  0.31 │  0.807  0.750  0.094  0.813   0.001   0.007 │  0.875  0.069  0.830   0.001 │    2/7
   0.10  0.32 │  0.810  0.750  0.508  0.813   0.037   0.024 │  0.750  0.673  0.830   0.054 │    3/7
   0.10  0.33 │  0.810  0.750  0.512  0.813   0.078   0.024 │  0.750  0.676  0.830   0.103 │    3/7
   0.10  0.34 │  0.811  0.750  0.513  0.813   0.113   0.024 │  0.750  0.676  0.830   0.150 │    3/7
   0.10  0.35 │  0.810  0.625  0.513  0.813   0.132

In [ ]:
# # ============================================================
# # CHECK STEP 1 + STEP 2 THRESHOLDS ON TRAIN SET
# # ============================================================

# import numpy as np
# import torch
# from sklearn.metrics import recall_score, precision_score, f1_score

# # ── Collect train probabilities ───────────────────────────
# model.eval()
# all_s1_probs_tr  = []
# all_s2_logits_tr = []
# all_y_orig_tr    = []

# with torch.no_grad():
#     for X, y1, y2, y_orig in train_loader:
#         s1, s2, _ = model(X)
#         all_s1_probs_tr.append(torch.sigmoid(s1).squeeze().cpu())
#         all_s2_logits_tr.append(s2.cpu())
#         all_y_orig_tr.append(y_orig.cpu())

# all_s1_probs_tr  = torch.cat(all_s1_probs_tr).numpy()
# all_s2_logits_tr = torch.cat(all_s2_logits_tr).numpy()
# all_y_orig_tr    = torch.cat(all_y_orig_tr).numpy()

# print(f"Train samples  : {len(all_s1_probs_tr):,}")
# print(f"Train sabotage : {(all_y_orig_tr==2).sum():,}")
# print(f"Train exfil    : {(all_y_orig_tr==1).sum():,}")
# print(f"Train normal   : {(all_y_orig_tr==0).sum():,}")

# threat_mask_tr = (all_y_orig_tr > 0)
# normal_mask_tr = (all_y_orig_tr == 0)
# sab_mask_tr    = (all_y_orig_tr == 2)
# exfil_mask_tr  = (all_y_orig_tr == 1)

# # ── Step 1 probability distribution ──────────────────────
# print(f"\n{'='*55}")
# print("STEP 1 PROBABILITY DISTRIBUTION")
# print(f"{'='*55}")
# print(f"  {'Group':<20} {'Mean':>8} {'Median':>8} "
#       f"{'Min':>8} {'Max':>8} {'>=0.5':>8}")
# print(f"  {'-'*55}")

# for name, mask in [
#     ("Normal",    normal_mask_tr),
#     ("Exfil",     exfil_mask_tr),
#     ("Sabotage",  sab_mask_tr),
#     ("All threat",threat_mask_tr),
# ]:
#     probs = all_s1_probs_tr[mask]
#     if len(probs) == 0:
#         continue
#     print(f"  {name:<20} "
#           f"{probs.mean():>8.4f} "
#           f"{np.median(probs):>8.4f} "
#           f"{probs.min():>8.4f} "
#           f"{probs.max():>8.4f} "
#           f"{(probs>=0.5).sum():>8,}")

# # ── Step 1 flagged count at each threshold ────────────────
# print(f"\n{'='*55}")
# print("STEP 1 — FLAGGED COUNT AT EACH THRESHOLD")
# print(f"{'='*55}")
# print(f"  {'t1':>6} {'flagged':>10} {'true_threat':>12} "
#       f"{'false_alarm':>12} {'sab_rec':>9} {'exfil_rec':>10}")
# print(f"  {'-'*62}")

# for t1 in np.round(np.arange(0.05, 0.55, 0.05), 2):
#     flagged     = all_s1_probs_tr >= t1
#     n_flagged   = flagged.sum()
#     true_threat = (flagged & threat_mask_tr).sum()
#     false_alarm = (flagged & normal_mask_tr).sum()
#     sab_rec     = (flagged & sab_mask_tr).sum()   / max(1, sab_mask_tr.sum())
#     exfil_rec   = (flagged & exfil_mask_tr).sum() / max(1, exfil_mask_tr.sum())
#     print(f"  {t1:>6.2f} {n_flagged:>10,} {true_threat:>12,} "
#           f"{false_alarm:>12,} {sab_rec:>9.4f} {exfil_rec:>10.4f}")

# # ── Step 2 probability distribution ──────────────────────
# threat_indices_tr  = np.where(threat_mask_tr)[0]
# s2_probs_tr        = torch.softmax(
#     torch.tensor(all_s2_logits_tr[threat_indices_tr]), dim=1
# ).numpy()

# true_sab_in_threat  = (all_y_orig_tr[threat_indices_tr] == 2)
# true_exfil_in_threat = (all_y_orig_tr[threat_indices_tr] == 1)

# print(f"\n{'='*55}")
# print("STEP 2 PROBABILITY DISTRIBUTION (true threat seqs only)")
# print(f"{'='*55}")
# print(f"  {'Group':<20} {'Mean P(sab)':>12} {'Median':>8} "
#       f"{'Min':>8} {'Max':>8}")
# print(f"  {'-'*55}")

# for name, mask in [
#     ("True Exfil",    true_exfil_in_threat),
#     ("True Sabotage", true_sab_in_threat),
# ]:
#     probs = s2_probs_tr[mask, 1]
#     if len(probs) == 0:
#         continue
#     print(f"  {name:<20} "
#           f"{probs.mean():>12.4f} "
#           f"{np.median(probs):>8.4f} "
#           f"{probs.min():>8.4f} "
#           f"{probs.max():>8.4f}")

# # ── Step 2 sabotage + exfil recall at each t2 ────────────
# print(f"\n{'='*55}")
# print("STEP 2 — SABOTAGE + EXFIL RECALL AT EACH t2")
# print("(applied to true threat sequences only)")
# print(f"{'='*55}")
# print(f"  {'t2':>6} {'sab_pred':>10} {'sab_rec':>9} "
#       f"{'exfil_pred':>11} {'exfil_rec':>10}")
# print(f"  {'-'*50}")

# n_true_sab   = true_sab_in_threat.sum()
# n_true_exfil = true_exfil_in_threat.sum()

# for t2 in np.round(np.arange(0.10, 0.65, 0.05), 2):
#     pred_sab   = (s2_probs_tr[:, 1] >= t2)
#     pred_exfil = ~pred_sab

#     sab_rec   = (pred_sab   & true_sab_in_threat).sum()  / max(1, n_true_sab)
#     exfil_rec = (pred_exfil & true_exfil_in_threat).sum()/ max(1, n_true_exfil)

#     print(f"  {t2:>6.2f} {pred_sab.sum():>10,} {sab_rec:>9.4f} "
#           f"{pred_exfil.sum():>11,} {exfil_rec:>10.4f}")

Train samples  : 552,331
Train sabotage : 448
Train exfil    : 8,076
Train normal   : 543,807

STEP 1 PROBABILITY DISTRIBUTION
  Group                    Mean   Median      Min      Max    >=0.5
  -------------------------------------------------------
  Normal                 0.0226   0.0095   0.0010   0.9999      675
  Exfil                  0.9567   1.0000   0.0097   1.0000    7,721
  Sabotage               0.9935   1.0000   0.0697   1.0000      445
  All threat             0.9587   1.0000   0.0097   1.0000    8,166

STEP 1 — FLAGGED COUNT AT EACH THRESHOLD
      t1    flagged  true_threat  false_alarm   sab_rec  exfil_rec
  --------------------------------------------------------------
    0.05     61,025        8,519       52,506    1.0000     0.9994
    0.10     28,024        8,504       19,520    0.9978     0.9976
    0.15     18,652        8,477       10,175    0.9978     0.9943
    0.20     14,548        8,442        6,106    0.9978     0.9900
    0.25     12,406        8,408 

COMMNENTS